# Hidden Markov Model (HMM) Analysis
### Master Thesis: Revisiting Delegation Theory in the Age of AI: Dynamic Algorithm Appreciation and Aversion in Triadic Organizational Relationships

---

### Table of Contents

**Part I — Data and HMM Estimation Framework**

1. [Data Description and Variable Construction](#1-data-description-and-variable-construction)
   - 1.1 &nbsp; [Helpers & Imports](#11-helpers--imports)
   - 1.2 &nbsp; [Benchmark Construction](#12-benchmark-construction)
   - 1.3 &nbsp; [Data Loading & Inspection](#13-data-loading--inspection)
2. [Hidden Markov Model Specification](#2-hidden-markov-model-specification)
   - 2.1 &nbsp; [Model Structure & Forward–Backward Algorithm](#21-model-structure--22-forwardbackward-algorithm)
   - 2.2 &nbsp; [Maximum Likelihood Estimation](#22-maximum-likelihood-estimation)
3. [Model Selection and Parameter Estimation](#3-model-selection-and-parameter-estimation)
4. [Posterior State Inference and Interpretation](#4-posterior-state-inference-and-interpretation)
   - 4.1 &nbsp; [Posterior State Assignment](#41-posterior-state-assignment)
   - 4.2 &nbsp; [Results Visualisation](#42-results-visualisation)

**Part II — Empirical Findings**

5. [KPI-Driven Transition Dynamics](#5-kpi-driven-transition-dynamics)
6. [Transparency as Moderator of State Transitions](#6-transparency-as-moderator-of-state-transitions)
7. [Strategic Control Retention Under High Task Stakes](#7-strategic-control-retention-under-high-task-stakes)

**Part III — Theoretical and Task-Level Implications**

8. [Delegation as Dynamic Learning Process](#8-delegation-as-dynamic-learning-process)
9. [Authority as Strategic and Legitimacy Mechanism](#9-authority-as-strategic-and-legitimacy-mechanism)
10. [Triadic Delegation Flows: Authority vs Execution](#10-triadic-delegation-flows-authority-vs-execution)
11. [Temporal Evolution of Delegation Patterns](#11-temporal-evolution-of-delegation-patterns)

**Appendix**

12. [Cluster Bootstrap Standard Errors](#12-cluster-bootstrap-standard-errors)
13. [Model Summary Table](#13-model-summary-table)

---
# Part I — Data and HMM Estimation Framework

<a id="1-data-description-and-variable-construction"></a>
## 1. Data Description and Variable Construction
<a id="11-helpers--imports"></a>
### 1.1 Helpers & Imports

In [6]:
# ============================================================
# 1. Imports & Helpers
# ============================================================
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from multiprocessing.pool import ThreadPool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

# ---- resolve data path ----
MANUAL_XLSX_PATH = None
PREFERRED_DATASETS = [
    Path(r"..\new run\Triadic_Delegation_Dataset_SYNTH_ANALYSIS_v2.xlsx"),
    Path(r"..\triadic_simulation\data\Triadic_Delegation_Dataset_SYNTH_ANALYSIS.xlsx"),
]
DATA_PATH = None
if MANUAL_XLSX_PATH:
    DATA_PATH = Path(MANUAL_XLSX_PATH)
else:
    for p in PREFERRED_DATASETS:
        if p.exists():
            DATA_PATH = p
            break
assert DATA_PATH is not None and DATA_PATH.exists(), \
    f"Data file not found. Tried: {PREFERRED_DATASETS}"
print(f"Data file: {DATA_PATH.resolve()}")


def softmax(z, axis=-1):
    """Numerically stable softmax (works on 1-D vectors and row-wise on 2-D)."""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    """Numerically stable log-softmax."""
    return z - logsumexp(z, axis=axis, keepdims=True)


def log_gaussian_diag(y, mean, log_sigma):
    """Scalar version (kept for reference)."""
    sigma2 = np.exp(2 * log_sigma)
    return -0.5 * (
        np.sum(np.log(2 * np.pi * sigma2))
        + np.sum((y - mean) ** 2 / sigma2)
    )

print("Section 1 — Imports & helpers loaded.")


Data file: C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\new run\Triadic_Delegation_Dataset_SYNTH_ANALYSIS_v2.xlsx
Section 1 — Imports & helpers loaded.


<a id="12-benchmark-construction"></a>
### 1.2 Benchmark Construction
Derive performance benchmarks used as **transition covariates** (P = 4):
- `kpi_operational_gap_index` — composite operational KPIs gap (intermediate)
- `within_unit_temporal_benchmark` — period-over-period KPI change, **mean-centred**
- `horizontal_peer_benchmark` — cross-sectional percentile rank within period
- `threshold_benchmark` — recent negative shock indicator (binary)
- `transparency_level_norm` — transparency main effect (normalised 0–1)

> `within_unit_ai_trajectory_benchmark` trimmed (high VIF). Transparency × performance interactions excluded in this specification.


In [7]:
# ============================================================
# 2. Build Benchmarks
# ============================================================

def build_benchmarks(df):
    df = df.sort_values(["manager_id", "period_id"]).copy()

    # Composite performance index (higher = better)
    df["kpi_operational_gap_index"] = (
        (-1.0) * df["service_level_delta"]
        + (-0.6) * df["inventory_cost_delta"]
        + (-0.4) * df["expedite_cost_delta"]
        + (-1.2) * df["error_incident_count"]
    )

    # Within-manager temporal benchmark (period-over-period diff, mean-centred)
    df["within_unit_temporal_benchmark"] = (
        df.groupby("manager_id")["kpi_operational_gap_index"].diff(1)
    )
    temp_mean = df["within_unit_temporal_benchmark"].mean()
    df["within_unit_temporal_benchmark"] = (
        df["within_unit_temporal_benchmark"] - temp_mean
    )

    # Peer percentile (cross-sectional)
    pct = df.groupby("period_id")["kpi_operational_gap_index"].rank(pct=True)
    df["horizontal_peer_benchmark"] = (pct - 0.5) * 2.0

    # Threshold shock (binary)
    df["threshold_benchmark"] = df["recent_negative_shock"].astype(float)

    # ── Transparency main effect ─────────────────────────────────────────────
    df["transparency_level_norm"] = df["transparency_level"].astype(float) / 3.0

    return df

print("Section 2 — build_benchmarks() defined.")


Section 2 — build_benchmarks() defined.


<a id="13-data-loading--inspection"></a>
### 1.3 Data Loading & Inspection
Load the `panel_manager_period` sheet, build benchmarks, scale variables, and form per-manager sequences.

In [9]:
# ============================================================
# 3. Data Loader
# ============================================================

@dataclass
class HMMData:
    Y: List[np.ndarray]      # emission sequences
    X: List[np.ndarray]      # transition covariates
    Z: List[np.ndarray]      # emission controls
    ids: List[str]
    periods: List[np.ndarray]
    y_scaler: StandardScaler
    x_scaler: StandardScaler
    z_scaler: StandardScaler


def load_sequences(xlsx_path):
    df = pd.read_excel(xlsx_path, sheet_name="panel_manager_period")

    # Safety: analysis file should NOT contain latent truth columns
    forbidden = ["latent_state_true", "latent_state_true_next"]
    if any(c in df.columns for c in forbidden):
        df = df.drop(columns=[c for c in forbidden if c in df.columns])
        print("  Dropped latent truth columns to proceed with analysis data.")

    df = build_benchmarks(df)

    # ── Compute share_authority_esc from decision_episode ──
    dec_ep = pd.read_excel(xlsx_path, sheet_name="decision_episode")
    dec_ep["is_escalated"] = (dec_ep["escalation_flag"] == 1).astype(int)
    esc_agg = (dec_ep
        .groupby(["manager_id", "period_id"])
        .agg(n_tasks=("episode_id", "count"),
             n_escalated=("is_escalated", "sum"))
        .reset_index())
    esc_agg["share_authority_esc"] = esc_agg["n_escalated"] / esc_agg["n_tasks"]
    df = df.merge(esc_agg[["manager_id", "period_id", "share_authority_esc"]],
                  on=["manager_id", "period_id"], how="left")
    df["share_authority_esc"] = df["share_authority_esc"].fillna(0.0)
    print(f"  share_authority_esc computed from decision_episode "
          f"(mean={df['share_authority_esc'].mean():.4f}, "
          f"std={df['share_authority_esc'].std():.4f})")

    # D=2: AI authority share + manager escalation share
    emission_cols = ["ai_decision_authority_share", "share_authority_esc"]

    # P=4: 3 benchmarks + transparency main effect only (no interactions).
    # within_unit_ai_trajectory_benchmark trimmed (high VIF).
    transition_cols = [
        "within_unit_temporal_benchmark",   # mean-centred period-over-period KPI diff
        "horizontal_peer_benchmark",        # cross-sectional percentile rank
        "threshold_benchmark",              # recent negative shock indicator
        "transparency_level_norm",          # transparency main effect (normalised 0–1)
    ]

    control_cols = [
        "task_complexity_index",
        "demand_volatility",
        "supply_disruption_count",
        "forecast_accuracy_mape",
        "decision_latency_avg",
        "target_difficulty",
        "performance_pressure_index",
        "recent_negative_shock",
    ]

    df = df.dropna(subset=emission_cols + transition_cols + control_cols)

    Y_list, X_list, Z_list = [], [], []
    ids, periods = [], []

    for mid, g in df.groupby("manager_id"):
        g = g.sort_values("period_id")
        Y = g[emission_cols].to_numpy(float)
        X = g[transition_cols].to_numpy(float)
        Z = g[control_cols].to_numpy(float)
        if len(Y) < 3:
            continue
        Y_list.append(Y)
        X_list.append(X)
        Z_list.append(Z)
        ids.append(mid)
        periods.append(g["period_id"].to_numpy())

    y_scaler = StandardScaler().fit(np.vstack(Y_list))
    x_scaler = StandardScaler().fit(np.vstack(X_list))
    z_scaler = StandardScaler().fit(np.vstack(Z_list))

    Y_list = [y_scaler.transform(y) for y in Y_list]
    X_list = [x_scaler.transform(x) for x in X_list]
    Z_list = [z_scaler.transform(z) for z in Z_list]

    return HMMData(Y_list, X_list, Z_list, ids, periods,
                   y_scaler, x_scaler, z_scaler)


# ---- Load & inspect ----
data = load_sequences(DATA_PATH)

seq_lens = [len(y) for y in data.Y]
print(f"Managers loaded : {len(data.Y)}")
print(f"Total observations: {sum(seq_lens)}")
print(f"Sequence lengths : min={min(seq_lens)}, median={int(np.median(seq_lens))}, max={max(seq_lens)}")
print(f"Emission dims (D) : {data.Y[0].shape[1]}")
print(f"Trans. covars (P) : {data.X[0].shape[1]}  "
      f"(3 benchmarks + 1 transparency main effect = 4)")
print(f"Controls (K)      : {data.Z[0].shape[1]}")


  share_authority_esc computed from decision_episode (mean=0.3508, std=0.0731)
Managers loaded : 120
Total observations: 3000
Sequence lengths : min=25, median=25, max=25
Emission dims (D) : 2
Trans. covars (P) : 4  (3 benchmarks + 1 transparency main effect = 4)
Controls (K)      : 8


<a id="2-hidden-markov-model-specification"></a>
## 2. Hidden Markov Model Specification
<a id="21-model-structure--22-forwardbackward-algorithm"></a>
### 2.1 Model Structure & 2.2 Forward–Backward Algorithm

In [10]:
# ============================================================
# Pre-stack all sequences to 3-D tensors for batched estimation
# ============================================================
import numpy as np

Y_stack = np.stack(data.Y)   # (N, T, D)
X_stack = np.stack(data.X)   # (N, T, P)
Z_stack = np.stack(data.Z)   # (N, T, K)

N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = N * T

print(f"Stacked arrays:")
print(f"  Y_stack: {Y_stack.shape}  (N={N}, T={T}, D={D})")
print(f"  X_stack: {X_stack.shape}  (N={N}, T={T}, P={P})")
print(f"  Z_stack: {Z_stack.shape}  (N={N}, T={T}, K={K})")
print(f"  n_obs_total = {n_obs_total}")

Stacked arrays:
  Y_stack: (120, 25, 2)  (N=120, T=25, D=2)
  X_stack: (120, 25, 4)  (N=120, T=25, P=4)
  Z_stack: (120, 25, 8)  (N=120, T=25, K=8)
  n_obs_total = 3000


In [11]:
# ── X collinearity check ─────────────────────────────────────────────────────
transition_cols_check = [
    "within_unit_temporal_benchmark",
    "horizontal_peer_benchmark",
    "threshold_benchmark",
    "transparency_level_norm",
]

X_flat = X_stack.reshape(-1, X_stack.shape[-1])   # (N*T, 4)
corr = np.corrcoef(X_flat.T)
df_corr = pd.DataFrame(corr, index=transition_cols_check, columns=transition_cols_check)

def color_high(val):
    return "background-color: #ff4444; color: white" if abs(val) >= 0.80 and abs(val) < 1.0 else ""

display(df_corr.round(3).style.map(color_high))

print("\nPairs with |r| >= 0.70:")
found = False
for i in range(len(transition_cols_check)):
    for j in range(i+1, len(transition_cols_check)):
        r = corr[i, j]
        if abs(r) >= 0.70:
            print(f"  {transition_cols_check[i]:40s}  {transition_cols_check[j]:40s}  r={r:.3f}")
            found = True
if not found:
    print("  None — all pairs below 0.70 ✓")


,within_unit_temporal_benchmark,horizontal_peer_benchmark,threshold_benchmark,transparency_level_norm
within_unit_temporal_benchmark,1.000000,0.446000,-0.473000,-0.000000
horizontal_peer_benchmark,0.446000,1.000000,-0.521000,-0.000000
threshold_benchmark,-0.473000,-0.521000,1.000000,0.010000
transparency_level_norm,-0.000000,-0.000000,0.010000,1.000000



Pairs with |r| >= 0.70:
  None — all pairs below 0.70 ✓


In [12]:
# ============================================================
# 4. Parameters + Forward–Backward  (VECTORIZED)
# ============================================================

@dataclass
class Params:
    logit_pi: np.ndarray   # (J,)
    alpha: np.ndarray      # (J, J)
    beta: np.ndarray       # (J, J, P)
    mu: np.ndarray         # (J, D)
    W: np.ndarray          # (J, D, K)
    log_sigma: np.ndarray  # (J, D)


def _precompute(p, Y, X, Z):
    """Shared emission + transition pre-computation."""
    T, D = Y.shape
    J = p.mu.shape[0]
    means = p.mu[None, :, :] + np.einsum('jdk,tk->tjd', p.W, Z)
    residuals = Y[:, None, :] - means
    sigma2 = np.exp(2 * p.log_sigma)
    log_norm = np.sum(np.log(2 * np.pi * sigma2), axis=1)
    logB = -0.5 * (log_norm[None, :] +
                   np.sum(residuals ** 2 / sigma2[None, :, :], axis=2))
    logits_all = (p.alpha[None, :, :]
                  + np.einsum('ijp,tp->tij', p.beta, X))
    logQ_all = log_softmax(logits_all, axis=2)
    return T, J, logB, logQ_all


def forward_only(p, Y, X, Z):
    """Forward pass only — returns log-likelihood (no posterior). ~2× faster."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    return float(logsumexp(log_alpha[-1]))


def forward_backward(p, Y, X, Z):
    """Full forward–backward returning (ll, log_gamma)."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    ll = logsumexp(log_alpha[-1])

    log_beta = np.zeros((T, J))
    for t in reversed(range(T - 1)):
        log_beta[t] = logsumexp(
            logQ_all[t + 1] + logB[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1)

    log_gamma = log_alpha + log_beta
    log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
    return ll, log_gamma

print("Section 4 — Params, forward_only() & forward_backward() defined.")

Section 4 — Params, forward_only() & forward_backward() defined.


<a id="22-maximum-likelihood-estimation"></a>
### 2.2 Maximum Likelihood Estimation
Estimate models with 2–4 latent states via maximum likelihood using L-BFGS-B optimization. To mitigate local optima, we employ multiple random initializations and diagonal-biased transition logits. Model selection follows a two-stage procedure: a computationally efficient screening phase identifies the most promising state specification using BIC, followed by a high-precision refit of the selected model using extended iterations and warm-start initialization. The final model is chosen based on the lowest Bayesian Information Criterion.

In [13]:
# ============================================================
# 5. Batched MLE Estimator  (fit_model_batched)
# ============================================================
# Accepts Y_stack, X_stack, Z_stack as EXPLICIT parameters so
# that Stack dimensions are always consistent with what was passed.
# Supports do_emission_only_warmstart for a two-phase init.
# ============================================================

import time
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp, log_softmax


def fit_model_batched(
    J: int,
    Y_stack: np.ndarray,
    X_stack: np.ndarray,
    Z_stack: np.ndarray,
    *,
    maxiter: int = 600,
    n_starts: int = 5,
    seed: int = 7,
    sigma_min: float = 0.05,
    sigma_max: float = 5.0,
    time_cap_min: int = 15,
    print_every: int = 50,
    l2: float = 1e-4,
    diag_bias: float = 2.0,
    maxfun: int = 200_000,
    ftol: float = 1e-8,
    gtol: float = 5e-6,
    warm_starts: list | None = None,
    use_subset: bool = False,
    subset_size: int = 80,
    do_emission_only_warmstart: bool = True,
    emission_only_maxiter: int = 120,
    emission_only_maxfun: int = 60_000,
):
    """
    Batched NH-HMM fit with Gaussian emissions and covariate-dependent
    transitions via L-BFGS-B.

    Parameters
    ----------
    Y_stack : (N, T, D) emission data
    X_stack : (N, T, P) transition covariates
    Z_stack : (N, T, K) emission controls
    do_emission_only_warmstart : if True, first optimize emission params only
        (mu, W, log_sigma) with transitions fixed, then use as init.
    """
    # Derive dimensions from passed stacks
    N_full, T, D = Y_stack.shape
    P = X_stack.shape[2]
    K = Z_stack.shape[2]

    # Optional subset for speed
    if use_subset and N_full > subset_size:
        rng_sub = np.random.default_rng(seed)
        idx = rng_sub.choice(N_full, size=subset_size, replace=False)
        Y_use = Y_stack[idx]
        X_use = X_stack[idx]
        Z_use = Z_stack[idx]
        N_use = subset_size
    else:
        Y_use, X_use, Z_use = Y_stack, X_stack, Z_stack
        N_use = N_full

    # ── pack / unpack ──
    def pack(p):
        return np.concatenate([
            p.logit_pi.ravel(), p.alpha.ravel(), p.beta.ravel(),
            p.mu.ravel(), p.W.ravel(), p.log_sigma.ravel(),
        ])

    def unpack(theta):
        idx = 0
        def take(n):
            nonlocal idx; v = theta[idx:idx+n]; idx += n; return v
        return Params(
            logit_pi=take(J),
            alpha=take(J*J).reshape(J,J),
            beta=take(J*J*P).reshape(J,J,P),
            mu=take(J*D).reshape(J,D),
            W=take(J*D*K).reshape(J,D,K),
            log_sigma=take(J*D).reshape(J,D),
        )

    # ── bounds (log_sigma only) ──
    n_logit_pi = J
    n_alpha = J*J
    n_beta = J*J*P
    n_mu = J*D
    n_W = J*D*K
    n_log_sigma = J*D
    log_sigma_start = n_logit_pi + n_alpha + n_beta + n_mu + n_W
    log_sigma_end = log_sigma_start + n_log_sigma
    total_params = log_sigma_end

    LOW, HIGH = np.log(sigma_min), np.log(sigma_max)
    bounds = [(None, None)] * total_params
    for i in range(log_sigma_start, log_sigma_end):
        bounds[i] = (LOW, HIGH)

    # ── smart init ──
    Y_flat = Y_use.reshape(-1, D)
    y_mean = Y_flat.mean(axis=0)
    y_std = np.maximum(Y_flat.std(axis=0), 1e-3)

    def smart_init_params(rng):
        mu0 = y_mean[None,:] + rng.normal(0, 1.0, (J,D)) * y_std[None,:]
        log_sigma0 = np.log(np.clip(y_std, sigma_min, sigma_max))[None,:]
        log_sigma0 = np.repeat(log_sigma0, J, axis=0)
        log_sigma0 = np.clip(log_sigma0 + rng.normal(0, 0.12, (J,D)), LOW, HIGH)
        logit_pi0 = rng.normal(0, 0.2, J)
        alpha0 = rng.normal(0, 0.20, (J,J)) + np.eye(J) * diag_bias
        beta0 = rng.normal(0, 0.02, (J,J,P))
        W0 = rng.normal(0, 0.03, (J,D,K))
        return Params(logit_pi=logit_pi0, alpha=alpha0, beta=beta0,
                      mu=mu0, W=W0, log_sigma=log_sigma0)

    # ── neg-LL (batched forward algorithm) ──
    stop_flag = {"stop": False}

    def neg_ll(theta):
        if stop_flag["stop"]:
            return 1e50
        p = unpack(theta)
        log_pi = log_softmax(p.logit_pi, axis=0)
        means = p.mu[None,None,:,:] + np.einsum("jdk,ntk->ntjd", p.W, Z_use)
        resid = Y_use[:,:,None,:] - means
        sigma2 = np.maximum(np.exp(2.0 * p.log_sigma), 1e-6)
        log_norm = np.sum(np.log(2*np.pi * sigma2), axis=1)
        logB = -0.5 * (log_norm[None,None,:] +
                       np.sum(resid**2 / sigma2[None,None,:,:], axis=3))
        logQ = log_softmax(
            p.alpha[None,None,:,:] + np.einsum("ijp,ntp->ntij", p.beta, X_use),
            axis=3)
        la = log_pi[None,:] + logB[:,0,:]
        for t in range(1, T):
            la = logB[:,t,:] + logsumexp(la[:,:,None] + logQ[:,t,:,:], axis=1)
        ll = np.sum(logsumexp(la, axis=1))
        return -float(ll) if np.isfinite(ll) else 1e40

    def objective(theta):
        base = neg_ll(theta)
        if not np.isfinite(base) or l2 <= 0:
            return base if np.isfinite(base) else 1e40
        p = unpack(theta)
        pen = (np.sum(p.alpha**2) + np.sum(p.beta**2) +
               np.sum(p.W**2) + 0.10*np.sum(p.mu**2))
        return base + l2 * pen

    # ── emission-only warmstart objective ──
    def emission_only_objective(em_theta, fixed_logit_pi, fixed_alpha, fixed_beta):
        """Optimize only mu, W, log_sigma with transitions frozen."""
        idx = 0
        def take(n):
            nonlocal idx; v = em_theta[idx:idx+n]; idx += n; return v
        mu = take(J*D).reshape(J,D)
        W = take(J*D*K).reshape(J,D,K)
        log_sigma = take(J*D).reshape(J,D)
        p = Params(logit_pi=fixed_logit_pi, alpha=fixed_alpha,
                   beta=fixed_beta, mu=mu, W=W, log_sigma=log_sigma)
        theta_full = pack(p)
        return objective(theta_full)

    # ── multi-start optimization ──
    init_list = list(warm_starts) if warm_starts else []
    runs = []

    for s in range(n_starts):
        rng = np.random.default_rng(seed + s)
        stop_flag["stop"] = False

        # Initialize
        if s < len(init_list):
            p0 = init_list[s]
            # Ensure dimensions match
            if p0.W.shape != (J, D, K):
                print(f"  ⚠️  start {s+1}: warm start W shape {p0.W.shape} "
                      f"!= ({J},{D},{K}), using random init")
                p0 = smart_init_params(rng)
            else:
                p0 = Params(
                    logit_pi=p0.logit_pi + rng.normal(0, 0.03, J),
                    alpha=p0.alpha + rng.normal(0, 0.03, (J,J)),
                    beta=p0.beta + rng.normal(0, 0.008, (J,J,P)),
                    mu=p0.mu + rng.normal(0, 0.05, (J,D)),
                    W=p0.W + rng.normal(0, 0.01, (J,D,K)),
                    log_sigma=np.clip(p0.log_sigma + rng.normal(0, 0.02, (J,D)),
                                      LOW, HIGH),
                )
        else:
            p0 = smart_init_params(rng)

        # Phase 1 (optional): emission-only warmstart
        if do_emission_only_warmstart and s >= len(init_list):
            em_theta0 = np.concatenate([
                p0.mu.ravel(), p0.W.ravel(), p0.log_sigma.ravel()])
            em_bounds = ([(None,None)]*(J*D + J*D*K) +
                         [(LOW,HIGH)]*(J*D))
            em_res = minimize(
                emission_only_objective, em_theta0,
                args=(p0.logit_pi, p0.alpha, p0.beta),
                method="L-BFGS-B", bounds=em_bounds,
                options={"maxiter": emission_only_maxiter,
                         "maxfun": emission_only_maxfun})
            # unpack emission params back
            eidx = 0
            def etake(n):
                nonlocal eidx; v = em_res.x[eidx:eidx+n]; eidx += n; return v
            p0 = Params(logit_pi=p0.logit_pi, alpha=p0.alpha,
                        beta=p0.beta,
                        mu=etake(J*D).reshape(J,D),
                        W=etake(J*D*K).reshape(J,D,K),
                        log_sigma=etake(J*D).reshape(J,D))

        # Phase 2: full optimization
        theta0 = pack(p0)
        start_time = time.time()
        iter_counter = {"i": 0}

        def callback(_xk):
            iter_counter["i"] += 1
            if iter_counter["i"] % print_every == 0:
                elapsed_min = (time.time() - start_time) / 60
                print(f"    J={J} start {s+1}/{n_starts} "
                      f"iter={iter_counter['i']} elapsed={elapsed_min:.1f} min",
                      flush=True)
            if (time.time() - start_time) > time_cap_min * 60:
                stop_flag["stop"] = True

        res = minimize(objective, theta0, method="L-BFGS-B",
                       bounds=bounds, callback=callback,
                       options={"maxiter": maxiter, "maxfun": maxfun,
                                "ftol": ftol, "gtol": gtol})

        if stop_flag["stop"]:
            res.success = False
            res.message = f"Time cap reached ({time_cap_min} min)"

        true_negll = neg_ll(res.x)
        runs.append((unpack(res.x), res, true_negll))
        print(f"    done: J={J} start {s+1}/{n_starts} success={res.success} "
              f"nit={getattr(res,'nit',None)} true_negLL={true_negll:.2f} "
              f"msg={res.message}", flush=True)

    # ── pick best run ──
    converged = [(p,r,tnl) for (p,r,tnl) in runs if bool(r.success)]
    if converged:
        best_p, best_res, best_tnl = min(converged, key=lambda t: t[2])
        best_is_conv = True
    else:
        best_p, best_res, best_tnl = min(runs, key=lambda t: t[2])
        best_is_conv = False

    best_res.true_negll = best_tnl
    best_res.true_ll = -best_tnl
    best_res.k_params = len(best_res.x)
    return best_p, best_res, best_is_conv


print("fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.")

fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.


In [8]:
# ============================================================
# STAGE 1: SCREENING  (J = 2, 3, 4)
# Goal:
#   1) Subset warm-start for each J (Stage 1A)
#   2) Full-data fit for each J using warm start (Stage 1B)
#   3) Select best J by BIC
# ============================================================

import time
import numpy as np
import pandas as pd

# ----------------------------
# Helper: build configs by J
# ----------------------------
def make_cfg_stage1a(J: int) -> dict:
    """Subset screening configs to generate warm start."""
    if J == 2:
        return dict(
            maxiter=600,
            n_starts=6,
            time_cap_min=15,
            diag_bias=2.0,
            maxfun=220000,
            use_subset=False,
            subset_size=80,
            l2=0.01,
            ftol=1e-7,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=140,
        )
    if J == 3:
        return dict(
            maxiter=900,
            n_starts=10,
            time_cap_min=25,
            diag_bias=2.6,
            maxfun=380000,
            use_subset=True,
            subset_size=95,
            l2=0.01,
            ftol=1e-7,
            gtol=2e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=180,
        )
    # J == 4  — raised caps for convergence with D=2 model
    return dict(
        maxiter=1500,
        n_starts=14,
        time_cap_min=60,
        diag_bias=2.9,
        maxfun=800000,
        use_subset=True,
        subset_size=95,
        l2=0.01,
        ftol=1e-7,
        gtol=2e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=260,
    )


def make_cfg_stage1b(J: int) -> dict:
    """Full-data screening configs seeded with subset warm starts."""
    if J == 2:
        return dict(
            maxiter=700,
            n_starts=6,
            time_cap_min=18,
            diag_bias=2.0,
            maxfun=260000,
            use_subset=False,
            l2=0.01,
            ftol=1e-7,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=140,
        )
    if J == 3:
        return dict(
            maxiter=1200,
            n_starts=10,
            time_cap_min=32,
            diag_bias=2.4,
            maxfun=520000,
            use_subset=False,
            l2=0.01,
            ftol=1e-7,
            gtol=2e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=200,
        )
    # J == 4  — raised caps for convergence with D=2 model
    return dict(
        maxiter=2000,
        n_starts=14,
        time_cap_min=80,
        diag_bias=2.7,
        maxfun=1000000,
        use_subset=False,
        l2=0.01,
        ftol=1e-7,
        gtol=2e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=320,
    )


# ----------------------------
# Stage 1A: Subset screening  (J = 2, 3, 4)
# ----------------------------
warm_by_J: dict[int, object] = {}
subset_log = []

print("\n=== STAGE 1A: Subset Screening — J = 2, 3, 4 ===")

for J in [2, 3, 4]:
    cfg = make_cfg_stage1a(J)

    print(f"\n[SubsetScreen] Fitting J={J} ...", flush=True)
    t0 = time.time()

    p_hat_sub, res_sub, is_conv_sub = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=7,
        sigma_min=0.1, sigma_max=3.5,
        print_every=50,
        **cfg
    )

    elapsed = time.time() - t0
    warm_by_J[J] = p_hat_sub

    subset_log.append({
        "stage": "subset",
        "J": J,
        "soft_converged": bool(is_conv_sub),
        "scipy_success": bool(getattr(res_sub, "success", False)),
        "LL": float(getattr(res_sub, "true_ll", np.nan)),
        "time_s": elapsed,
        "message": str(getattr(res_sub, "message", "")),
    })

    print(
        f"[SubsetScreen] J={J} soft_converged={bool(is_conv_sub)} "
        f"LL={float(getattr(res_sub,'true_ll',np.nan)):.1f} "
        f"({elapsed/60:.1f} min) msg={getattr(res_sub,'message','')}",
        flush=True
    )

df_subset = pd.DataFrame(subset_log)
display(df_subset.round(2))


# ----------------------------
# Stage 1B: Full-data — J = 2, 3, 4
# ----------------------------
screen_results = []

print("\n=== STAGE 1B: Full-data Fit — J = 2, 3, 4 (warm-started) ===")

for J in [2, 3, 4]:
    cfg_full = make_cfg_stage1b(J)

    warm = warm_by_J.get(J)
    if warm is None:
        print(f"⚠️  No warm start for J={J}. Skipping full-data screening.")
        continue

    print(f"\n[ScreenFull] Fitting J={J} (warm-started) ...", flush=True)
    t0 = time.time()

    p_hat, res, is_conv = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=77,
        sigma_min=0.1, sigma_max=3.5,
        print_every=50,
        warm_starts=[warm],
        **cfg_full
    )

    elapsed = time.time() - t0

    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))

    bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
    aic = 2.0 * k_params - 2.0 * ll_total

    screen_results.append({
        "stage": "screen_full",
        "J": J,
        "params": k_params,
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "converged_soft": bool(is_conv),
        "converged_scipy": bool(getattr(res, "success", False)),
        "time_s": elapsed,
        "message": str(getattr(res, "message", "")),
    })

    sym = "✓" if bool(is_conv) else "✗"
    print(
        f"[ScreenFull] J={J} LL={ll_total:.1f} BIC={bic:.1f} params={k_params} "
        f"{sym} ({elapsed/60:.1f} min)"
    )
    print(f"            message: {getattr(res,'message','')}")

df_screen = pd.DataFrame(screen_results)
display(df_screen.round(2))


# ----------------------------
# Select best J by BIC
# ----------------------------
best_J_screen = int(df_screen.loc[df_screen["BIC"].idxmin(), "J"])
best_row = df_screen[df_screen["J"] == best_J_screen].iloc[0]
print(
    f"\n★ Best J by BIC: J={best_J_screen} "
    f"(BIC={best_row['BIC']:.1f}, soft_converged={bool(best_row['converged_soft'])})"
)
print(df_screen[["J", "params", "LL", "AIC", "BIC", "converged_soft"]].to_string(index=False))



=== STAGE 1A: Subset Screening — J = 2, 3, 4 ===

[SubsetScreen] Fitting J=2 ...
    J=2 start 1/6 iter=50 elapsed=1.0 min
    J=2 start 1/6 iter=100 elapsed=2.0 min
    J=2 start 1/6 iter=150 elapsed=2.9 min
    J=2 start 1/6 iter=200 elapsed=3.6 min
    J=2 start 1/6 iter=250 elapsed=4.3 min
    done: J=2 start 1/6 success=True nit=285 true_negLL=1708.35 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/6 iter=50 elapsed=0.8 min
    J=2 start 2/6 iter=100 elapsed=1.4 min
    J=2 start 2/6 iter=150 elapsed=1.9 min
    J=2 start 2/6 iter=200 elapsed=3.2 min
    J=2 start 2/6 iter=250 elapsed=5.2 min
    done: J=2 start 2/6 success=True nit=286 true_negLL=1708.32 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 3/6 iter=50 elapsed=1.1 min
    J=2 start 3/6 iter=100 elapsed=2.2 min
    J=2 start 3/6 iter=150 elapsed=3.3 min
    J=2 start 3/6 iter=200 elapsed=4.4 min
    J=2 start 3/6 iter=250 elapsed=5.5 min
    J=2 start 3/6 iter=300 elapsed

,stage,J,soft_converged,scipy_success,LL,time_s,message
0,subset,2,True,True,-1708.32,1950.39,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
1,subset,3,False,False,-21.03,9506.28,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
2,subset,4,True,True,724.80,34713.77,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



=== STAGE 1B: Full-data Fit — J = 2, 3, 4 (warm-started) ===

[ScreenFull] Fitting J=2 (warm-started) ...
    J=2 start 1/6 iter=50 elapsed=0.4 min
    done: J=2 start 1/6 success=True nit=85 true_negLL=1708.33 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/6 iter=50 elapsed=0.4 min
    J=2 start 2/6 iter=100 elapsed=0.7 min
    J=2 start 2/6 iter=150 elapsed=1.1 min
    J=2 start 2/6 iter=200 elapsed=1.4 min
    done: J=2 start 2/6 success=True nit=248 true_negLL=1708.35 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 3/6 iter=50 elapsed=0.4 min
    J=2 start 3/6 iter=100 elapsed=0.8 min
    J=2 start 3/6 iter=150 elapsed=1.1 min
    J=2 start 3/6 iter=200 elapsed=1.5 min
    done: J=2 start 3/6 success=True nit=241 true_negLL=1708.37 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 4/6 iter=50 elapsed=0.4 min
    J=2 start 4/6 iter=100 elapsed=0.7 min
    J=2 start 4/6 iter=150 elapsed=1.1 min
    done: J=2 start

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,screen_full,2,62,-1708.33,3540.65,3913.05,True,True,748.95,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
1,screen_full,3,108,84.31,47.39,696.07,False,False,11713.02,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
2,screen_full,4,164,964.64,-1601.29,-616.24,True,True,38063.78,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



★ Best J by BIC: J=4 (BIC=-616.2, soft_converged=True)
 J  params           LL          AIC         BIC  converged_soft
 2      62 -1708.326221  3540.652442 3913.047231            True
 3     108    84.307118    47.385765  696.073462           False
 4     164   964.644423 -1601.288845 -616.244564            True


## Rescued refit for non-converged J=3

In [14]:
# ============================================================
# J = 3 ONLY: SCREEN + FULL FIT + RESCUE
# Goal:
#   1) Subset warm-start for J=3
#   2) Full-data fit for J=3
#   3) If needed, run dedicated rescue retries
#   4) Keep best converged J=3 result
# ============================================================

import time
import copy
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configs for J = 3
# ------------------------------------------------------------
def make_cfg_j3_stage1a() -> dict:
    """Subset screening config for warm start."""
    return dict(
        maxiter=1200,
        n_starts=12,
        time_cap_min=35,
        diag_bias=2.9,
        maxfun=500000,
        use_subset=True,
        subset_size=110,
        l2=0.008,
        ftol=5e-8,
        gtol=1e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=240,
    )


def make_cfg_j3_stage1b() -> dict:
    """Full-data config using subset warm start."""
    return dict(
        maxiter=1600,
        n_starts=12,
        time_cap_min=45,
        diag_bias=2.8,
        maxfun=700000,
        use_subset=False,
        l2=0.008,
        ftol=5e-8,
        gtol=1e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=260,
    )


def make_cfg_j3_rescue() -> dict:
    """Extra-strong rescue config if Stage 1B does not converge."""
    return dict(
        maxiter=2200,
        n_starts=16,
        time_cap_min=75,
        diag_bias=3.0,
        maxfun=900000,
        use_subset=False,
        l2=0.005,
        ftol=5e-8,
        gtol=1e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=320,
    )


# ------------------------------------------------------------
# Helper: summarize a run
# ------------------------------------------------------------
def summarize_fit(stage_name, J, res, is_conv, elapsed_s):
    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))
    bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
    aic = 2.0 * k_params - 2.0 * ll_total

    return {
        "stage": stage_name,
        "J": J,
        "params": k_params,
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "converged_soft": bool(is_conv),
        "converged_scipy": bool(getattr(res, "success", False)),
        "time_s": elapsed_s,
        "message": str(getattr(res, "message", "")),
    }


# ------------------------------------------------------------
# Helper: pick best row with preference for soft convergence
# ------------------------------------------------------------
def choose_best_candidate(candidates):
    """
    Prefer:
      1) soft-converged
      2) lower BIC
    """
    if not candidates:
        return None

    scored = []
    for c in candidates:
        penalty = 0.0 if bool(c["converged_soft"]) else 1e6
        scored.append((c["BIC"] + penalty, c))

    scored.sort(key=lambda x: x[0])
    return scored[0][1]


# ------------------------------------------------------------
# Stage 1A: subset warm start for J=3
# ------------------------------------------------------------
J = 3
cfg_stage1a = make_cfg_j3_stage1a()

print("\n=== J=3 ONLY — STAGE 1A: SUBSET WARM START ===", flush=True)
print("Config:", cfg_stage1a, flush=True)

t0 = time.time()
p_hat_sub_j3, res_sub_j3, is_conv_sub_j3 = fit_model_batched(
    J=J,
    Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
    seed=7,
    sigma_min=0.08,
    sigma_max=4.0,
    print_every=50,
    **cfg_stage1a
)
elapsed_sub = time.time() - t0

row_sub = summarize_fit("subset_j3", J, res_sub_j3, is_conv_sub_j3, elapsed_sub)
df_sub = pd.DataFrame([row_sub])
display(df_sub.round(2))

print(
    f"[Subset J=3] soft_converged={bool(is_conv_sub_j3)} "
    f"LL={row_sub['LL']:.1f} BIC={row_sub['BIC']:.1f} "
    f"({elapsed_sub/60:.1f} min)"
)
print(f"             message: {getattr(res_sub_j3, 'message', '')}")


# ------------------------------------------------------------
# Stage 1B: full-data fit for J=3 using warm start
# ------------------------------------------------------------
cfg_stage1b = make_cfg_j3_stage1b()
warm_j3 = p_hat_sub_j3 if p_hat_sub_j3 is not None else None
warm_list_j3 = [warm_j3] if warm_j3 is not None else None

print("\n=== J=3 ONLY — STAGE 1B: FULL-DATA FIT ===", flush=True)
print("Config:", cfg_stage1b, flush=True)

t0 = time.time()
p_hat_full_j3, res_full_j3, is_conv_full_j3 = fit_model_batched(
    J=J,
    Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
    seed=77,
    sigma_min=0.08,
    sigma_max=4.0,
    print_every=50,
    warm_starts=warm_list_j3,
    **cfg_stage1b
)
elapsed_full = time.time() - t0

row_full = summarize_fit("full_j3", J, res_full_j3, is_conv_full_j3, elapsed_full)
df_full = pd.DataFrame([row_full])
display(df_full.round(2))

print(
    f"[Full J=3] soft_converged={bool(is_conv_full_j3)} "
    f"LL={row_full['LL']:.1f} BIC={row_full['BIC']:.1f} "
    f"({elapsed_full/60:.1f} min)"
)
print(f"            message: {getattr(res_full_j3, 'message', '')}")


# ------------------------------------------------------------
# Rescue: if full-data J=3 does not converge
# ------------------------------------------------------------
rescue_rows = []
rescue_objects = []

if not bool(is_conv_full_j3):
    cfg_rescue = make_cfg_j3_rescue()

    print("\n=== J=3 ONLY — RESCUE RETRIES ===", flush=True)
    print("Rescue config:", cfg_rescue, flush=True)

    rescue_seeds = [777, 888, 999, 1234, 2027, 3031]

    for k, seed in enumerate(rescue_seeds, start=1):
        print(f"\n[J3 Rescue] Attempt {k}/{len(rescue_seeds)} | seed={seed}", flush=True)

        t0 = time.time()
        p_hat_try, res_try, is_conv_try = fit_model_batched(
            J=J,
            Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
            seed=seed,
            sigma_min=0.08,
            sigma_max=4.0,
            print_every=50,
            warm_starts=warm_list_j3,
            **cfg_rescue
        )
        elapsed_try = time.time() - t0

        row_try = summarize_fit("rescue_j3", J, res_try, is_conv_try, elapsed_try)
        row_try["seed"] = seed
        rescue_rows.append(row_try)

        rescue_objects.append({
            "seed": seed,
            "p_hat": p_hat_try,
            "res": res_try,
            "is_conv": is_conv_try,
            "row": row_try,
        })

        print(
            f"[J3 Rescue] seed={seed} "
            f"soft_converged={bool(is_conv_try)} "
            f"LL={row_try['LL']:.1f} BIC={row_try['BIC']:.1f} "
            f"({elapsed_try/60:.1f} min)"
        )
        print(f"             message: {getattr(res_try, 'message', '')}")

    df_rescue = pd.DataFrame(rescue_rows)
    display(df_rescue.round(2))
else:
    print("\n✓ Full-data J=3 already soft-converged. Rescue not needed.")


# ------------------------------------------------------------
# Choose best final J=3 result
# ------------------------------------------------------------
candidate_rows = [row_full] + rescue_rows
best_row_j3 = choose_best_candidate(candidate_rows)

if best_row_j3 is None:
    raise RuntimeError("No J=3 candidate results found.")

# map row back to fitted object
if best_row_j3["stage"] == "full_j3":
    best_p_hat_J3 = p_hat_full_j3
    best_res_J3 = res_full_j3
    best_is_conv_J3 = is_conv_full_j3
    best_cfg_J3 = copy.deepcopy(cfg_stage1b)
else:
    best_obj = None
    for obj in rescue_objects:
        if (
            obj["row"]["seed"] == best_row_j3["seed"]
            and obj["row"]["BIC"] == best_row_j3["BIC"]
        ):
            best_obj = obj
            break

    if best_obj is None:
        raise RuntimeError("Could not map best rescue row back to fitted object.")

    best_p_hat_J3 = best_obj["p_hat"]
    best_res_J3 = best_obj["res"]
    best_is_conv_J3 = best_obj["is_conv"]
    best_cfg_J3 = copy.deepcopy(make_cfg_j3_rescue())

# store selection variables consistent with your pipeline naming
best_J_screen = 3

print("\n=== FINAL SELECTED J=3 RESULT ===")
print(
    f"J={best_J_screen} | "
    f"LL={best_row_j3['LL']:.1f} | "
    f"AIC={best_row_j3['AIC']:.1f} | "
    f"BIC={best_row_j3['BIC']:.1f} | "
    f"soft_converged={bool(best_row_j3['converged_soft'])} | "
    f"scipy_success={bool(best_row_j3['converged_scipy'])}"
)
print(f"message: {best_row_j3['message']}")
print(f"selected stage: {best_row_j3['stage']}")

df_all_j3 = pd.DataFrame(candidate_rows)
display(df_all_j3.round(2))

# optional compact summary
print("\nAll J=3 attempts:")
print(
    df_all_j3[
        ["stage", "J", "params", "LL", "AIC", "BIC", "converged_soft", "converged_scipy", "time_s"]
    ].to_string(index=False)
)


=== J=3 ONLY — STAGE 1A: SUBSET WARM START ===
Config: {'maxiter': 1200, 'n_starts': 12, 'time_cap_min': 35, 'diag_bias': 2.9, 'maxfun': 500000, 'use_subset': True, 'subset_size': 110, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 240}


    J=3 start 1/12 iter=50 elapsed=0.8 min
    J=3 start 1/12 iter=100 elapsed=1.4 min
    J=3 start 1/12 iter=150 elapsed=2.1 min
    J=3 start 1/12 iter=200 elapsed=2.8 min
    J=3 start 1/12 iter=250 elapsed=3.5 min
    J=3 start 1/12 iter=300 elapsed=4.1 min
    J=3 start 1/12 iter=350 elapsed=4.9 min
    J=3 start 1/12 iter=400 elapsed=5.6 min
    J=3 start 1/12 iter=450 elapsed=6.3 min
    J=3 start 1/12 iter=500 elapsed=7.0 min
    J=3 start 1/12 iter=550 elapsed=7.7 min
    J=3 start 1/12 iter=600 elapsed=8.3 min
    J=3 start 1/12 iter=650 elapsed=9.0 min
    J=3 start 1/12 iter=700 elapsed=9.7 min
    J=3 start 1/12 iter=750 elapsed=10.4 min
    J=3 start 1/12 iter=800 elapsed=11.1 min
    J=3 start 1/12 iter=850 elapsed=11.7 min
    J=3 start 1/12 iter=900 elapsed=12.4 min
    J=3 start 1/12 iter=950 elapsed=13.1 min
    J=3 start 1/12 iter=1000 elapsed=13.7 min
    J=3 start 1/12 iter=1050 elapsed=14.4 min
    J=3 start 1/12 iter=1100 elapsed=15.0 min
    J=3 start 1/12 ite

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,subset_j3,3,108,-1004.77,2225.54,2874.23,True,True,13542.53,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Subset J=3] soft_converged=True LL=-1004.8 BIC=2874.2 (225.7 min)
             message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== J=3 ONLY — STAGE 1B: FULL-DATA FIT ===
Config: {'maxiter': 1600, 'n_starts': 12, 'time_cap_min': 45, 'diag_bias': 2.8, 'maxfun': 700000, 'use_subset': False, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 260}
    J=3 start 1/12 iter=50 elapsed=0.8 min
    J=3 start 1/12 iter=100 elapsed=1.6 min
    J=3 start 1/12 iter=150 elapsed=2.4 min
    J=3 start 1/12 iter=200 elapsed=3.3 min
    J=3 start 1/12 iter=250 elapsed=4.0 min
    J=3 start 1/12 iter=300 elapsed=4.9 min
    J=3 start 1/12 iter=350 elapsed=5.6 min
    J=3 start 1/12 iter=400 elapsed=6.4 min
    done: J=3 start 1/12 success=True nit=421 true_negLL=1111.08 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=3 start 2/12 iter=50 elapsed=0.8 min
    J=3 start 2/12 iter=100 elapsed=1.6 min
    J=3 start 2/12 iter=15

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j3,3,108,-225.86,667.72,1316.41,True,True,15780.26,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Full J=3] soft_converged=True LL=-225.9 BIC=1316.4 (263.0 min)
            message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

✓ Full-data J=3 already soft-converged. Rescue not needed.

=== FINAL SELECTED J=3 RESULT ===
J=3 | LL=-225.9 | AIC=667.7 | BIC=1316.4 | soft_converged=True | scipy_success=True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
selected stage: full_j3


,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j3,3,108,-225.86,667.72,1316.41,True,True,15780.26,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



All J=3 attempts:
  stage  J  params          LL        AIC         BIC  converged_soft  converged_scipy       time_s
full_j3  3     108 -225.861997 667.723995 1316.411692            True             True 15780.262179


## Adjust J=2 and J=4 to similar setting with rescued refit J=3 (for publishable cross-J selection purposes)

In [28]:
# ============================================================
# MATCHED RE-FIT PIPELINE FOR J = 2 AND J = 4
# Goal:
#   1) Run stronger, comparable estimation for J=2 and J=4
#   2) Keep best converged result per J
#   3) Combine with your rescued J=3 for fairer cross-J selection
# ============================================================

import time
import copy
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Helper: summarize a run
# ------------------------------------------------------------
def summarize_fit(stage_name, J, res, is_conv, elapsed_s, seed=None):
    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))
    bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
    aic = 2.0 * k_params - 2.0 * ll_total

    row = {
        "stage": stage_name,
        "J": J,
        "params": k_params,
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "converged_soft": bool(is_conv),
        "converged_scipy": bool(getattr(res, "success", False)),
        "time_s": elapsed_s,
        "message": str(getattr(res, "message", "")),
    }
    if seed is not None:
        row["seed"] = seed
    return row


# ------------------------------------------------------------
# Helper: choose best candidate
# Prefer:
#   1) soft-converged
#   2) lower BIC
# ------------------------------------------------------------
def choose_best_candidate(candidates):
    if not candidates:
        return None

    scored = []
    for c in candidates:
        penalty = 0.0 if bool(c["converged_soft"]) else 1e6
        scored.append((c["BIC"] + penalty, c))

    scored.sort(key=lambda x: x[0])
    return scored[0][1]


# ------------------------------------------------------------
# Config builders
# These are made broadly parallel to your rescued J=3 setup
# while allowing modest complexity differences by J.
# ------------------------------------------------------------
def make_cfg_refit_stage1a(J: int) -> dict:
    """
    Subset warm-start config.
    Similar spirit to the new J=3 Stage 1A.
    """
    if J == 2:
        return dict(
            maxiter=1000,
            n_starts=10,
            time_cap_min=30,
            diag_bias=2.6,
            maxfun=420000,
            use_subset=True,
            subset_size=110,
            l2=0.008,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=220,
        )
    elif J == 4:
        return dict(
            maxiter=1400,
            n_starts=14,
            time_cap_min=40,
            diag_bias=3.0,
            maxfun=650000,
            use_subset=True,
            subset_size=110,
            l2=0.008,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=280,
        )
    else:
        raise ValueError("This pipeline is for J=2 and J=4 only.")


def make_cfg_refit_stage1b(J: int) -> dict:
    """
    Full-data fit config.
    Similar spirit to the new J=3 Stage 1B.
    """
    if J == 2:
        return dict(
            maxiter=1400,
            n_starts=10,
            time_cap_min=40,
            diag_bias=2.5,
            maxfun=600000,
            use_subset=False,
            l2=0.008,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=240,
        )
    elif J == 4:
        return dict(
            maxiter=1800,
            n_starts=14,
            time_cap_min=55,
            diag_bias=2.9,
            maxfun=850000,
            use_subset=False,
            l2=0.008,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=320,
        )
    else:
        raise ValueError("This pipeline is for J=2 and J=4 only.")


def make_cfg_refit_rescue(J: int) -> dict:
    """
    Rescue config if Stage 1B does not converge.
    Similar spirit to J=3 rescue.
    """
    if J == 2:
        return dict(
            maxiter=1800,
            n_starts=14,
            time_cap_min=60,
            diag_bias=2.8,
            maxfun=800000,
            use_subset=False,
            l2=0.005,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=300,
        )
    elif J == 4:
        return dict(
            maxiter=2400,
            n_starts=18,
            time_cap_min=85,
            diag_bias=3.1,
            maxfun=1100000,
            use_subset=False,
            l2=0.005,
            ftol=5e-8,
            gtol=1e-5,
            do_emission_only_warmstart=True,
            emission_only_maxiter=380,
        )
    else:
        raise ValueError("This pipeline is for J=2 and J=4 only.")


# ------------------------------------------------------------
# Main function: run matched refit for a single J
# ------------------------------------------------------------
def run_matched_refit_for_J(J: int):
    print(f"\n{'='*70}")
    print(f"MATCHED RE-FIT PIPELINE FOR J={J}")
    print(f"{'='*70}")

    # -------------------------
    # Stage 1A: subset warm start
    # -------------------------
    cfg_stage1a = make_cfg_refit_stage1a(J)
    print("\n--- Stage 1A: Subset warm start ---")
    print("Config:", cfg_stage1a)

    t0 = time.time()
    p_hat_sub, res_sub, is_conv_sub = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=7,
        sigma_min=0.08,
        sigma_max=4.0,
        print_every=50,
        **cfg_stage1a
    )
    elapsed_sub = time.time() - t0

    row_sub = summarize_fit(
        stage_name=f"subset_j{J}",
        J=J,
        res=res_sub,
        is_conv=is_conv_sub,
        elapsed_s=elapsed_sub
    )
    display(pd.DataFrame([row_sub]).round(2))

    print(
        f"[Subset J={J}] soft_converged={bool(is_conv_sub)} "
        f"LL={row_sub['LL']:.1f} BIC={row_sub['BIC']:.1f} "
        f"({elapsed_sub/60:.1f} min)"
    )
    print(f"               message: {getattr(res_sub, 'message', '')}")

    warm = p_hat_sub if p_hat_sub is not None else None
    warm_list = [warm] if warm is not None else None

    # -------------------------
    # Stage 1B: full-data fit
    # -------------------------
    cfg_stage1b = make_cfg_refit_stage1b(J)
    print("\n--- Stage 1B: Full-data fit ---")
    print("Config:", cfg_stage1b)

    t0 = time.time()
    p_hat_full, res_full, is_conv_full = fit_model_batched(
        J=J,
        Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
        seed=77,
        sigma_min=0.08,
        sigma_max=4.0,
        print_every=50,
        warm_starts=warm_list,
        **cfg_stage1b
    )
    elapsed_full = time.time() - t0

    row_full = summarize_fit(
        stage_name=f"full_j{J}",
        J=J,
        res=res_full,
        is_conv=is_conv_full,
        elapsed_s=elapsed_full
    )
    display(pd.DataFrame([row_full]).round(2))

    print(
        f"[Full J={J}] soft_converged={bool(is_conv_full)} "
        f"LL={row_full['LL']:.1f} BIC={row_full['BIC']:.1f} "
        f"({elapsed_full/60:.1f} min)"
    )
    print(f"             message: {getattr(res_full, 'message', '')}")

    # -------------------------
    # Rescue if needed
    # -------------------------
    rescue_rows = []
    rescue_objects = []

    if not bool(is_conv_full):
        cfg_rescue = make_cfg_refit_rescue(J)
        print("\n--- Rescue retries ---")
        print("Config:", cfg_rescue)

        rescue_seeds = [777, 888, 999, 1234, 2027, 3031]

        for k, seed in enumerate(rescue_seeds, start=1):
            print(f"\n[Rescue J={J}] Attempt {k}/{len(rescue_seeds)} | seed={seed}")

            t0 = time.time()
            p_hat_try, res_try, is_conv_try = fit_model_batched(
                J=J,
                Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
                seed=seed,
                sigma_min=0.08,
                sigma_max=4.0,
                print_every=50,
                warm_starts=warm_list,
                **cfg_rescue
            )
            elapsed_try = time.time() - t0

            row_try = summarize_fit(
                stage_name=f"rescue_j{J}",
                J=J,
                res=res_try,
                is_conv=is_conv_try,
                elapsed_s=elapsed_try,
                seed=seed
            )
            rescue_rows.append(row_try)

            rescue_objects.append({
                "seed": seed,
                "p_hat": p_hat_try,
                "res": res_try,
                "is_conv": is_conv_try,
                "row": row_try,
            })

            print(
                f"[Rescue J={J}] seed={seed} "
                f"soft_converged={bool(is_conv_try)} "
                f"LL={row_try['LL']:.1f} BIC={row_try['BIC']:.1f} "
                f"({elapsed_try/60:.1f} min)"
            )
            print(f"               message: {getattr(res_try, 'message', '')}")

        display(pd.DataFrame(rescue_rows).round(2))
    else:
        print(f"\n✓ Full-data J={J} already soft-converged. Rescue not needed.")

    # -------------------------
    # Choose best candidate for this J
    # -------------------------
    candidate_rows = [row_full] + rescue_rows
    best_row = choose_best_candidate(candidate_rows)

    if best_row is None:
        raise RuntimeError(f"No candidate results found for J={J}.")

    if best_row["stage"] == f"full_j{J}":
        best_p_hat = p_hat_full
        best_res = res_full
        best_is_conv = is_conv_full
        best_cfg = copy.deepcopy(cfg_stage1b)
    else:
        best_obj = None
        for obj in rescue_objects:
            if (
                obj["row"].get("seed", None) == best_row.get("seed", None)
                and obj["row"]["BIC"] == best_row["BIC"]
            ):
                best_obj = obj
                break

        if best_obj is None:
            raise RuntimeError(f"Could not map best rescue row back to fitted object for J={J}.")

        best_p_hat = best_obj["p_hat"]
        best_res = best_obj["res"]
        best_is_conv = best_obj["is_conv"]
        best_cfg = copy.deepcopy(make_cfg_refit_rescue(J))

    print(f"\n=== FINAL BEST-CONVERGED RESULT FOR J={J} ===")
    print(
        f"J={J} | "
        f"LL={best_row['LL']:.1f} | "
        f"AIC={best_row['AIC']:.1f} | "
        f"BIC={best_row['BIC']:.1f} | "
        f"soft_converged={bool(best_row['converged_soft'])} | "
        f"scipy_success={bool(best_row['converged_scipy'])}"
    )
    print(f"message: {best_row['message']}")
    print(f"selected stage: {best_row['stage']}")

    df_all = pd.DataFrame(candidate_rows)
    display(df_all.round(2))

    print(f"\nAll J={J} attempts:")
    print(
        df_all[
            ["stage", "J", "params", "LL", "AIC", "BIC",
             "converged_soft", "converged_scipy", "time_s"]
        ].to_string(index=False)
    )

    return {
        "J": J,
        "best_row": best_row,
        "best_p_hat": best_p_hat,
        "best_res": best_res,
        "best_is_conv": best_is_conv,
        "best_cfg": best_cfg,
        "df_all": df_all,
    }


# ------------------------------------------------------------
# Run matched refits for J=2 and J=4
# ------------------------------------------------------------
out_j2 = run_matched_refit_for_J(2)
out_j4 = run_matched_refit_for_J(4)


MATCHED RE-FIT PIPELINE FOR J=2

--- Stage 1A: Subset warm start ---
Config: {'maxiter': 1000, 'n_starts': 10, 'time_cap_min': 30, 'diag_bias': 2.6, 'maxfun': 420000, 'use_subset': True, 'subset_size': 110, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 220}
    J=2 start 1/10 iter=50 elapsed=0.5 min
    J=2 start 1/10 iter=100 elapsed=0.9 min
    J=2 start 1/10 iter=150 elapsed=1.3 min
    J=2 start 1/10 iter=200 elapsed=1.8 min
    J=2 start 1/10 iter=250 elapsed=2.3 min
    done: J=2 start 1/10 success=True nit=278 true_negLL=1558.03 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/10 iter=50 elapsed=0.4 min
    J=2 start 2/10 iter=100 elapsed=0.8 min
    J=2 start 2/10 iter=150 elapsed=1.2 min
    J=2 start 2/10 iter=200 elapsed=1.6 min
    J=2 start 2/10 iter=250 elapsed=2.0 min
    J=2 start 2/10 iter=300 elapsed=2.4 min
    done: J=2 start 2/10 success=True nit=312 true_negLL=1558.04 msg=CONVERGENC

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,subset_j2,2,62,-1558.02,3240.04,3612.44,True,True,2006.38,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Subset J=2] soft_converged=True LL=-1558.0 BIC=3612.4 (33.4 min)
               message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

--- Stage 1B: Full-data fit ---
Config: {'maxiter': 1400, 'n_starts': 10, 'time_cap_min': 40, 'diag_bias': 2.5, 'maxfun': 600000, 'use_subset': False, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 240}
    J=2 start 1/10 iter=50 elapsed=0.4 min
    J=2 start 1/10 iter=100 elapsed=0.9 min
    done: J=2 start 1/10 success=True nit=148 true_negLL=1708.34 msg=CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
    J=2 start 2/10 iter=50 elapsed=0.3 min
    J=2 start 2/10 iter=100 elapsed=0.7 min
    J=2 start 2/10 iter=150 elapsed=1.0 min
    J=2 start 2/10 iter=200 elapsed=1.4 min
    J=2 start 2/10 iter=250 elapsed=1.9 min
    J=2 start 2/10 iter=300 elapsed=2.3 min
    J=2 start 2/10 iter=350 elapsed=2.7 min
    done: J=2 start 2/10 success=True nit=372 true_negLL=1708.32 msg=CONVERGENCE

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j2,2,62,-1708.32,3540.63,3913.03,True,True,1868.69,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Full J=2] soft_converged=True LL=-1708.3 BIC=3913.0 (31.1 min)
             message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

✓ Full-data J=2 already soft-converged. Rescue not needed.

=== FINAL BEST-CONVERGED RESULT FOR J=2 ===
J=2 | LL=-1708.3 | AIC=3540.6 | BIC=3913.0 | soft_converged=True | scipy_success=True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
selected stage: full_j2


...

[Subset J=4] soft_converged=False LL=943.9 BIC=-574.8 (545.5 min)
               message: STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

--- Stage 1B: Full-data fit ---
Config: {'maxiter': 1800, 'n_starts': 14, 'time_cap_min': 55, 'diag_bias': 2.9, 'maxfun': 850000, 'use_subset': False, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 320}
    J=4 start 1/14 iter=50 elapsed=1.4 min
    J=4 start 1/14 iter=100 elapsed=2.6 min
    J=4 start 1/14 iter=150 elapsed=4.0 min
    J=4 start 1/14 iter=200 elapsed=5.2 min
    J=4 start 1/14 iter=250 elapsed=6.5 min
    J=4 start 1/14 iter=300 elapsed=7.8 min
    J=4 start 1/14 iter=350 elapsed=9.1 min
    J=4 start 1/14 iter=400 elapsed=10.3 min
    J=4 start 1/14 iter=450 elapsed=11.6 min
    J=4 start 1/14 iter=500 elapsed=12.8 min
    J=4 start 1/14 iter=550 elapsed=14.0 min
    J=4 start 1/14 iter=600 elapsed=15.3 min
    J=4 start 1/14 iter=650 elapsed=16.5 min
    J=4 start 1/14 iter=700 

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j4,4,164,1008.82,-1689.64,-704.6,True,True,59976.47,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Full J=4] soft_converged=True LL=1008.8 BIC=-704.6 (999.6 min)
             message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

✓ Full-data J=4 already soft-converged. Rescue not needed.

=== FINAL BEST-CONVERGED RESULT FOR J=4 ===
J=4 | LL=1008.8 | AIC=-1689.6 | BIC=-704.6 | soft_converged=True | scipy_success=True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
selected stage: full_j4


,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j4,4,164,1008.82,-1689.64,-704.6,True,True,59976.47,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



All J=4 attempts:
  stage  J  params          LL          AIC         BIC  converged_soft  converged_scipy       time_s
full_j4  4     164 1008.821587 -1689.643175 -704.598893            True             True 59976.473428


## Rerun the model with J=4 to be comparable with J3 again

In [17]:

# ============================================================
# J = 4 ONLY: SCREEN + FULL FIT + RESCUE
# Goal:
#   1) Subset warm-start for J=4
#   2) Full-data fit for J=4
#   3) If needed, run dedicated rescue retries
#   4) Keep best converged J=4 result
# ============================================================

import time
import copy
import json
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd

def _ensure_fit_model_batched():
    """Resolve fit_model_batched from globals/aliases/notebook source."""
    fn = globals().get("fit_model_batched")
    if callable(fn):
        return fn

    for alias in (
        "fit_model",
        "fit_model_multi_start",
        "fit_model_with_restarts",
        "fit_model_batched_v2",
    ):
        cand = globals().get(alias)
        if callable(cand):
            globals()["fit_model_batched"] = cand
            return cand

    nb_path = Path(
        r"c:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\data_analysis\hmm_2_emissions_v3_4 covariates.ipynb"
    )
    if nb_path.exists():
        try:
            nb = json.loads(nb_path.read_text(encoding="utf-8"))
            ns = globals()
            for cell in nb.get("cells", []):
                if cell.get("cell_type") != "code":
                    continue
                src = "".join(cell.get("source", []))
                if "def fit_model_batched" in src:
                    exec(src, ns, ns)
                    break
        except Exception as e:
            print(f"[warn] Could not auto-load fit_model_batched: {e}")

    fn = globals().get("fit_model_batched")
    if callable(fn):
        return fn

    def _missing_fit_model_batched(*args, **kwargs):
        msg = (
            "fit_model_batched is not available. Run the cell that defines it, "
            "then re-run this cell."
        )
        print(f"[warn] {msg}")
        return None, SimpleNamespace(true_ll=np.nan, x=[], success=False, message=msg), False

    globals()["fit_model_batched"] = _missing_fit_model_batched
    return _missing_fit_model_batched

fit_model_batched = _ensure_fit_model_batched()


def _first_defined(*names):
    for n in names:
        if n in globals() and globals()[n] is not None:
            return globals()[n]
    return None


def _resolve_stacks():
    """
    Return (Y_stack, X_stack, Z_stack) as numpy arrays exactly as defined
    in the data-prep cell.  We simply look up common names; if the value is
    already a numpy ndarray with ndim >= 2 we return it as-is.
    """
    y_val = _first_defined("Y_stack", "Y", "Y_list", "Y_seq", "Y_batches")
    x_val = _first_defined("X_stack", "X", "X_list", "X_seq", "X_batches")
    z_val = _first_defined("Z_stack", "Z", "Z_list", "Z_seq", "Z_batches")

    missing = [n for n, v in [("Y_stack", y_val), ("X_stack", x_val), ("Z_stack", z_val)] if v is None]
    if missing:
        raise NameError(
            f"Missing required input(s): {', '.join(missing)}. "
            f"Run the data-prep cell first or define aliases (Y/X/Z)."
        )

    return y_val, x_val, z_val


def _run_fit_model_batched(**kwargs):
    out = fit_model_batched(**kwargs)

    if isinstance(out, tuple):
        if len(out) == 3:
            return out[0], out[1], out[2]
        if len(out) == 2:
            p_hat, res = out
            return p_hat, res, bool(getattr(res, "success", False))
        raise ValueError(f"Unexpected tuple length from fit_model_batched: {len(out)}")

    res = out
    return None, res, bool(getattr(res, "success", False))


Y_stack, X_stack, Z_stack = _resolve_stacks()

if "n_obs_total" not in globals() or globals().get("n_obs_total") is None:
    try:
        n_obs_total = int(np.asarray(Y_stack).shape[0] * np.asarray(Y_stack).shape[1])
    except Exception:
        n_obs_total = int(len(Y_stack))


# ------------------------------------------------------------
# Configs for J = 4
# ------------------------------------------------------------
def make_cfg_j4_stage1a() -> dict:
    """Subset screening config for warm start."""
    return dict(
        maxiter=1400,
        n_starts=14,
        time_cap_min=40,
        diag_bias=3.0,
        maxfun=650000,
        use_subset=True,
        subset_size=110,
        l2=0.008,
        ftol=5e-8,
        gtol=1e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=280,
    )


def make_cfg_j4_stage1b() -> dict:
    """Full-data config using subset warm start."""
    return dict(
        maxiter=1800,
        n_starts=14,
        time_cap_min=55,
        diag_bias=2.9,
        maxfun=850000,
        use_subset=False,
        l2=0.008,
        ftol=5e-8,
        gtol=1e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=320,
    )


def make_cfg_j4_rescue() -> dict:
    """Extra-strong rescue config if Stage 1B does not converge."""
    return dict(
        maxiter=2200,
        n_starts=8,
        time_cap_min=70,
        diag_bias=2.8,
        maxfun=1100000,
        use_subset=False,
        l2=0.006,
        ftol=1e-9,
        gtol=1e-6,
        do_emission_only_warmstart=True,
        emission_only_maxiter=400,
    )


# ------------------------------------------------------------
# Helper: summarize a run
# ------------------------------------------------------------
def summarize_fit(stage_name, J, res, is_conv, elapsed_s, seed=None):
    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))
    bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
    aic = 2.0 * k_params - 2.0 * ll_total

    row = {
        "stage": stage_name,
        "J": J,
        "params": k_params,
        "LL": ll_total,
        "AIC": aic,
        "BIC": bic,
        "converged_soft": bool(is_conv),
        "converged_scipy": bool(getattr(res, "success", False)),
        "time_s": elapsed_s,
        "message": str(getattr(res, "message", "")),
    }
    if seed is not None:
        row["seed"] = seed
    return row


# ------------------------------------------------------------
# Helper: choose best candidate
#   Priority:
#   1) soft-converged
#   2) lower BIC
# ------------------------------------------------------------
def choose_best_candidate(candidates):
    if not candidates:
        return None

    scored = []
    for c in candidates:
        penalty = 0.0 if bool(c["converged_soft"]) else 1e6
        scored.append((c["BIC"] + penalty, c))

    scored.sort(key=lambda x: x[0])
    return scored[0][1]


# ------------------------------------------------------------
# Stage 1A: subset warm start for J=4
# ------------------------------------------------------------
J = 4
cfg_stage1a = make_cfg_j4_stage1a()

print("\n=== J=4 ONLY — STAGE 1A: SUBSET WARM START ===", flush=True)
print("Config:", cfg_stage1a, flush=True)

t0 = time.time()
p_hat_sub_j4, res_sub_j4, is_conv_sub_j4 = _run_fit_model_batched(
    J=J,
    Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
    seed=7,
    sigma_min=0.08,
    sigma_max=4.0,
    print_every=50,
    **cfg_stage1a
)
elapsed_sub = time.time() - t0

row_sub = summarize_fit("sub_j4", J, res_sub_j4, is_conv_sub_j4, elapsed_sub)
df_sub = pd.DataFrame([row_sub])
display(df_sub.round(2))

print(
    f"[Sub J=4] soft_converged={bool(is_conv_sub_j4)} "
    f"LL={row_sub['LL']:.1f} BIC={row_sub['BIC']:.1f} "
    f"({elapsed_sub/60:.1f} min)"
)
print(f"             message: {getattr(res_sub_j4, 'message', '')}")


# ------------------------------------------------------------
# Stage 1B: full-data fit for J=4 using warm start
# ------------------------------------------------------------
cfg_stage1b = make_cfg_j4_stage1b()
warm_j4 = p_hat_sub_j4 if p_hat_sub_j4 is not None else None
warm_list_j4 = [warm_j4] if warm_j4 is not None else None

print("\n=== J=4 ONLY — STAGE 1B: FULL-DATA FIT ===", flush=True)
print("Config:", cfg_stage1b, flush=True)

t0 = time.time()
p_hat_full_j4, res_full_j4, is_conv_full_j4 = _run_fit_model_batched(
    J=J,
    Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
    seed=77,
    sigma_min=0.08,
    sigma_max=4.0,
    print_every=50,
    warm_starts=warm_list_j4,
    **cfg_stage1b
)
elapsed_full = time.time() - t0

row_full = summarize_fit("full_j4", J, res_full_j4, is_conv_full_j4, elapsed_full)
df_full = pd.DataFrame([row_full])
display(df_full.round(2))

print(
    f"[Full J=4] soft_converged={bool(is_conv_full_j4)} "
    f"LL={row_full['LL']:.1f} BIC={row_full['BIC']:.1f} "
    f"({elapsed_full/60:.1f} min)"
)
print(f"            message: {getattr(res_full_j4, 'message', '')}")


# ------------------------------------------------------------
# Rescue: if full-data J=4 does not converge
# ------------------------------------------------------------
rescue_rows = []
rescue_objects = []

if not bool(is_conv_full_j4):
    cfg_rescue = make_cfg_j4_rescue()

    print("\n=== J=4 ONLY — RESCUE RETRIES ===", flush=True)
    print("Rescue config:", cfg_rescue, flush=True)

    rescue_seeds = [777, 888, 999, 1234, 2027, 3031]

    for k, seed in enumerate(rescue_seeds, start=1):
        print(f"\n[J4 Rescue] Attempt {k}/{len(rescue_seeds)} | seed={seed}", flush=True)

        t0 = time.time()
        p_hat_try, res_try, is_conv_try = _run_fit_model_batched(
            J=J,
            Y_stack=Y_stack, X_stack=X_stack, Z_stack=Z_stack,
            seed=seed,
            sigma_min=0.08,
            sigma_max=4.0,
            print_every=50,
            warm_starts=warm_list_j4,
            **cfg_rescue
        )
        elapsed_try = time.time() - t0

        row_try = summarize_fit("rescue_j4", J, res_try, is_conv_try, elapsed_try, seed=seed)
        rescue_rows.append(row_try)

        rescue_objects.append({
            "seed": seed,
            "p_hat": p_hat_try,
            "res": res_try,
            "is_conv": is_conv_try,
            "row": row_try,
        })

        print(
            f"[J4 Rescue] seed={seed} "
            f"soft_converged={bool(is_conv_try)} "
            f"LL={row_try['LL']:.1f} BIC={row_try['BIC']:.1f} "
            f"({elapsed_try/60:.1f} min)"
        )
        print(f"             message: {getattr(res_try, 'message', '')}")

    df_rescue = pd.DataFrame(rescue_rows)
    display(df_rescue.round(2))
else:
    print("\n✓ Full-data J=4 already soft-converged. Rescue not needed.")


# ------------------------------------------------------------
# Choose best final J=4 result
# ------------------------------------------------------------
candidate_rows = [row_full] + rescue_rows
best_row_j4 = choose_best_candidate(candidate_rows)

if best_row_j4 is None:
    raise RuntimeError("No J=4 candidate results found.")

# map row back to fitted object
if best_row_j4["stage"] == "full_j4":
    best_p_hat_J4 = p_hat_full_j4
    best_res_J4 = res_full_j4
    best_is_conv_J4 = is_conv_full_j4
    best_cfg_J4 = copy.deepcopy(cfg_stage1b)
else:
    best_obj = None
    for obj in rescue_objects:
        if (
            obj["row"]["seed"] == best_row_j4.get("seed")
            and obj["row"]["BIC"] == best_row_j4["BIC"]
        ):
            best_obj = obj
            break

    if best_obj is None:
        raise RuntimeError("Could not map best rescue row back to fitted object.")

    best_p_hat_J4 = best_obj["p_hat"]
    best_res_J4 = best_obj["res"]
    best_is_conv_J4 = best_obj["is_conv"]
    best_cfg_J4 = copy.deepcopy(make_cfg_j4_rescue())

print("\n=== FINAL SELECTED J=4 RESULT ===")
print(
    f"J=4 | "
    f"LL={best_row_j4['LL']:.1f} | "
    f"AIC={best_row_j4['AIC']:.1f} | "
    f"BIC={best_row_j4['BIC']:.1f} | "
    f"soft_converged={bool(best_row_j4['converged_soft'])} | "
    f"scipy_success={bool(best_row_j4['converged_scipy'])}"
)
print(f"message: {best_row_j4['message']}")
print(f"selected stage: {best_row_j4['stage']}")

df_all_j4 = pd.DataFrame(candidate_rows)
display(df_all_j4.round(2))

print("\nAll J=4 attempts:")
print(
    df_all_j4[
        ["stage", "J", "params", "LL", "AIC", "BIC", "converged_soft", "converged_scipy", "time_s"]
    ].to_string(index=False)
)



=== J=4 ONLY — STAGE 1A: SUBSET WARM START ===
Config: {'maxiter': 1400, 'n_starts': 14, 'time_cap_min': 40, 'diag_bias': 3.0, 'maxfun': 650000, 'use_subset': True, 'subset_size': 110, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 280}
    J=4 start 1/14 iter=50 elapsed=1.3 min
    J=4 start 1/14 iter=100 elapsed=2.6 min
    J=4 start 1/14 iter=150 elapsed=4.1 min
    J=4 start 1/14 iter=200 elapsed=5.4 min
    J=4 start 1/14 iter=250 elapsed=6.8 min
    J=4 start 1/14 iter=300 elapsed=8.3 min
    J=4 start 1/14 iter=350 elapsed=9.8 min
    J=4 start 1/14 iter=400 elapsed=11.1 min
    J=4 start 1/14 iter=450 elapsed=12.5 min
    J=4 start 1/14 iter=500 elapsed=13.8 min
    J=4 start 1/14 iter=550 elapsed=15.2 min
    J=4 start 1/14 iter=600 elapsed=16.8 min
    J=4 start 1/14 iter=650 elapsed=18.1 min
    J=4 start 1/14 iter=700 elapsed=19.5 min
    J=4 start 1/14 iter=750 elapsed=20.7 min
    J=4 start 1/14 iter=800 elapsed=22

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,sub_j4,4,164,943.91,-1559.81,-574.77,False,False,59676.11,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT


[Sub J=4] soft_converged=False LL=943.9 BIC=-574.8 (994.6 min)
             message: STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

=== J=4 ONLY — STAGE 1B: FULL-DATA FIT ===
Config: {'maxiter': 1800, 'n_starts': 14, 'time_cap_min': 55, 'diag_bias': 2.9, 'maxfun': 850000, 'use_subset': False, 'l2': 0.008, 'ftol': 5e-08, 'gtol': 1e-05, 'do_emission_only_warmstart': True, 'emission_only_maxiter': 320}
    J=4 start 1/14 iter=50 elapsed=1.5 min
    J=4 start 1/14 iter=100 elapsed=2.9 min
    J=4 start 1/14 iter=150 elapsed=4.3 min
    J=4 start 1/14 iter=200 elapsed=5.7 min
    J=4 start 1/14 iter=250 elapsed=7.1 min
    J=4 start 1/14 iter=300 elapsed=8.6 min
    J=4 start 1/14 iter=350 elapsed=10.1 min
    J=4 start 1/14 iter=400 elapsed=11.5 min
    J=4 start 1/14 iter=450 elapsed=12.9 min
    J=4 start 1/14 iter=500 elapsed=14.3 min
    J=4 start 1/14 iter=550 elapsed=15.8 min
    J=4 start 1/14 iter=600 elapsed=17.2 min
    J=4 start 1/14 iter=650 elapsed=18.6 min
    J=4 start 1/14 it

,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j4,4,164,1008.82,-1689.64,-704.6,True,True,55234.56,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


[Full J=4] soft_converged=True LL=1008.8 BIC=-704.6 (920.6 min)
            message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

✓ Full-data J=4 already soft-converged. Rescue not needed.

=== FINAL SELECTED J=4 RESULT ===
J=4 | LL=1008.8 | AIC=-1689.6 | BIC=-704.6 | soft_converged=True | scipy_success=True
message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
selected stage: full_j4


,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j4,4,164,1008.82,-1689.64,-704.6,True,True,55234.56,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



All J=4 attempts:
  stage  J  params          LL          AIC         BIC  converged_soft  converged_scipy       time_s
full_j4  4     164 1008.821587 -1689.643175 -704.598893            True             True 55234.556781


In [22]:
# ============================================================
# FINAL MODEL SELECTION ACROSS J = 2, 3, 4
# Rule:
#   1) Keep only soft-converged models
#   2) Choose lowest BIC
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Collect final best row per J
# ------------------------------------------------------------
final_candidates = []

# J=2
if "out_j2" in globals() and out_j2 is not None:
    final_candidates.append(dict(out_j2["best_row"]))
else:
    print("Warning: out_j2 not found.")

# J=3
if "best_row_j3" in globals() and best_row_j3 is not None:
    final_candidates.append(dict(best_row_j3))
elif "out_j3" in globals() and out_j3 is not None:
    final_candidates.append(dict(out_j3["best_row"]))
else:
    print("Warning: best_row_j3 / out_j3 not found.")

# J=4
if "best_row_j4" in globals() and best_row_j4 is not None:
    final_candidates.append(dict(best_row_j4))
elif "out_j4" in globals() and out_j4 is not None:
    final_candidates.append(dict(out_j4["best_row"]))
else:
    print("Warning: best_row_j4 / out_j4 not found.")

# ------------------------------------------------------------
# Create comparison table
# ------------------------------------------------------------
df_final_compare = pd.DataFrame(final_candidates).copy()

if df_final_compare.empty:
    raise RuntimeError("No final candidates found for model selection.")

# keep a clean display order
preferred_cols = [
    "stage", "J", "params", "LL", "AIC", "BIC",
    "converged_soft", "converged_scipy", "time_s", "message"
]
existing_cols = [c for c in preferred_cols if c in df_final_compare.columns]
df_final_compare = df_final_compare[existing_cols].sort_values(["J"]).reset_index(drop=True)

print("\n=== FINAL CROSS-J COMPARISON ===")
display(df_final_compare.round(3))
print(df_final_compare.round(3).to_string(index=False))

# ------------------------------------------------------------
# Select best model:
# prefer soft-converged, then lowest BIC
# ------------------------------------------------------------
df_valid = df_final_compare[df_final_compare["converged_soft"] == True].copy()

if df_valid.empty:
    raise RuntimeError("No soft-converged models available for final selection.")

df_valid = df_valid.sort_values(["BIC", "AIC", "J"], ascending=[True, True, True]).reset_index(drop=True)
final_pick_row = df_valid.iloc[0].to_dict()

best_J = int(final_pick_row["J"])
best_BIC = float(final_pick_row["BIC"])
best_AIC = float(final_pick_row["AIC"])
best_LL = float(final_pick_row["LL"])
best_stage = final_pick_row["stage"]

print("\n=== FINAL SELECTED MODEL ===")
print(f"Best J: {best_J}")
print(f"Selected stage: {best_stage}")
print(f"LL:  {best_LL:.6f}")
print(f"AIC: {best_AIC:.6f}")
print(f"BIC: {best_BIC:.6f}")

# ------------------------------------------------------------
# Store final selection in pipeline-friendly objects
# ------------------------------------------------------------
final_pick = final_pick_row
best_J_screen = best_J

# ------------------------------------------------------------
# Optional: map best J back to fitted objects/configs
# ------------------------------------------------------------
if best_J == 2:
    best_row_final = out_j2["best_row"]
    best_p_hat_final = out_j2["best_p_hat"]
    best_res_final = out_j2["best_res"]
    best_is_conv_final = out_j2["best_is_conv"]
    final_cfg = out_j2["best_cfg"]

elif best_J == 3:
    best_row_final = best_row_j3
    best_p_hat_final = best_p_hat_J3
    best_res_final = best_res_J3
    best_is_conv_final = best_is_conv_J3
    final_cfg = best_cfg_J3

elif best_J == 4:
    # works whether you stored J=4 via out_j4 or via standalone J=4 block
    if "out_j4" in globals() and out_j4 is not None:
        best_row_final = out_j4["best_row"]
        best_p_hat_final = out_j4["best_p_hat"]
        best_res_final = out_j4["best_res"]
        best_is_conv_final = out_j4["best_is_conv"]
        final_cfg = out_j4["best_cfg"]
    else:
        best_row_final = best_row_j4
        best_p_hat_final = best_p_hat_J4
        best_res_final = best_res_J4
        best_is_conv_final = best_is_conv_J4
        final_cfg = best_cfg_J4

else:
    raise RuntimeError(f"Unexpected best_J={best_J}")

print("\nObjects saved:")
print(" - final_pick")
print(" - best_J_screen")
print(" - best_row_final")
print(" - best_p_hat_final")
print(" - best_res_final")
print(" - best_is_conv_final")
print(" - final_cfg")


=== FINAL CROSS-J COMPARISON ===


,stage,J,params,LL,AIC,BIC,converged_soft,converged_scipy,time_s,message
0,full_j4,4,164,1008.822,-1689.643,-704.599,True,True,40280.607,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


  stage  J  params       LL       AIC      BIC  converged_soft  converged_scipy    time_s                                              message
full_j4  4     164 1008.822 -1689.643 -704.599            True             True 40280.607 CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

=== FINAL SELECTED MODEL ===
Best J: 4
Selected stage: full_j4
LL:  1008.821587
AIC: -1689.643175
BIC: -704.598893

Objects saved:
 - final_pick
 - best_J_screen
 - best_row_final
 - best_p_hat_final
 - best_res_final
 - best_is_conv_final
 - final_cfg


## Diagnostics check to compare J3 and J4

In [18]:
# ============================================================
# DIAGNOSTIC TOOLKIT FOR J=3 AND J=4
# Assumes fitted objects like:
#   best_p_hat_J3, best_res_J3, best_row_j3
#   best_p_hat_J4, best_res_J4, best_row_j4
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _safe_getattr(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    return default

def _to_numpy(x):
    if x is None:
        return None
    try:
        return np.asarray(x)
    except Exception:
        return None

def _extract_gamma(model):
    """
    Try common names for posterior state probabilities gamma.
    Expected shape:
      (N, J) or (T, J) or list of arrays
    """
    cand = [
        "gamma", "Gamma", "posterior_probs", "posterior_probabilities",
        "smoothed_probs", "state_posteriors", "xi_gamma"
    ]
    g = _safe_getattr(model, cand, default=None)
    if g is None:
        return None

    if isinstance(g, list):
        try:
            g = np.vstack([np.asarray(a) for a in g])
        except Exception:
            return None

    g = _to_numpy(g)
    if g is None:
        return None
    return g

def _extract_transition_matrix(model):
    cand = ["A", "trans_mat", "transition_matrix", "P", "Q"]
    A = _safe_getattr(model, cand, default=None)
    A = _to_numpy(A)
    return A

def _extract_mu_sigma(model):
    """
    Try to extract state-specific emission means and sigmas.
    Expected:
      mu shape ~ (J, D)
      sigma shape ~ (J, D) or compatible
    """
    mu_names = ["mu", "Mu", "mus", "means", "emission_means"]
    sg_names = ["sigma", "Sigma", "sigmas", "std", "stds", "emission_sigmas"]

    mu = _safe_getattr(model, mu_names, default=None)
    sg = _safe_getattr(model, sg_names, default=None)

    mu = _to_numpy(mu)
    sg = _to_numpy(sg)

    return mu, sg

def _extract_W(model):
    """
    Emission control coefficients W.
    Expected maybe shape (J, D, K) or (D, J, K) depending on implementation.
    """
    cand = ["W", "w", "emission_coefs", "emission_betas"]
    W = _safe_getattr(model, cand, default=None)
    return _to_numpy(W)

def _extract_alpha_beta(model):
    """
    Transition intercepts / covariates if present.
    """
    alpha = _safe_getattr(model, ["alpha", "Alpha", "transition_intercepts"], default=None)
    beta  = _safe_getattr(model, ["beta", "Beta", "transition_betas"], default=None)
    return _to_numpy(alpha), _to_numpy(beta)

def posterior_summary(model, J):
    gamma = _extract_gamma(model)
    if gamma is None:
        return None, None, None, None

    if gamma.ndim != 2:
        return None, None, None, None

    if gamma.shape[1] != J and gamma.shape[0] == J:
        gamma = gamma.T

    if gamma.shape[1] != J:
        print(f"Warning: gamma shape {gamma.shape} does not match J={J}")
        return gamma, None, None, None

    occ = gamma.mean(axis=0)
    hard = gamma.argmax(axis=1)
    hard_share = np.bincount(hard, minlength=J) / len(hard)
    maxprob = gamma.max(axis=1)
    mean_certainty = float(maxprob.mean())

    entropy = -np.sum(gamma * np.log(np.clip(gamma, 1e-12, 1.0)), axis=1)
    mean_entropy = float(entropy.mean())

    return gamma, occ, hard_share, {
        "mean_posterior_certainty": mean_certainty,
        "mean_entropy": mean_entropy
    }

def emission_summary(model, J, emission_names=None):
    mu, sg = _extract_mu_sigma(model)

    if mu is None:
        return None

    mu = np.asarray(mu)
    if mu.ndim == 1:
        mu = mu.reshape(J, 1)

    if mu.shape[0] != J and mu.shape[1] == J:
        mu = mu.T

    if mu.shape[0] != J:
        print(f"Warning: mu shape {mu.shape} does not match J={J}")
        return None

    D = mu.shape[1]
    if emission_names is None or len(emission_names) != D:
        emission_names = [f"emission_{d+1}" for d in range(D)]

    # sigma
    if sg is not None:
        sg = np.asarray(sg)
        if sg.ndim == 1:
            if len(sg) == J:
                sg = sg.reshape(J, 1)
            elif len(sg) == D:
                sg = np.tile(sg.reshape(1, D), (J, 1))
        elif sg.ndim == 2:
            if sg.shape[0] != J and sg.shape[1] == J:
                sg = sg.T

        if sg.shape[0] != J:
            sg = None

    rows = []
    for j in range(J):
        for d in range(D):
            rows.append({
                "state": j + 1,
                "emission": emission_names[d],
                "mean": float(mu[j, d]),
                "sigma": float(sg[j, d]) if sg is not None and sg.ndim == 2 else np.nan,
            })

    return pd.DataFrame(rows)

def pairwise_state_distance(em_df):
    """
    Simple pairwise Euclidean distance using emission means only.
    """
    if em_df is None or em_df.empty:
        return None

    pivot = em_df.pivot(index="state", columns="emission", values="mean").sort_index()
    X = pivot.values
    states = pivot.index.tolist()

    rows = []
    for i in range(len(states)):
        for j in range(i + 1, len(states)):
            dist = float(np.linalg.norm(X[i] - X[j]))
            rows.append({
                "state_i": states[i],
                "state_j": states[j],
                "euclidean_distance_means": dist
            })
    return pd.DataFrame(rows).sort_values("euclidean_distance_means")

def summarize_model(label, J, model, res, row, emission_names=None):
    gamma, occ, hard_share, post_stats = posterior_summary(model, J)
    em_df = emission_summary(model, J, emission_names=emission_names)
    dist_df = pairwise_state_distance(em_df)
    A = _extract_transition_matrix(model)
    alpha, beta = _extract_alpha_beta(model)
    W = _extract_W(model)

    fit_row = pd.DataFrame([{
        "model": label,
        "J": J,
        "LL": float(row["LL"]),
        "AIC": float(row["AIC"]),
        "BIC": float(row["BIC"]),
        "soft_converged": bool(row["converged_soft"]),
        "scipy_success": bool(row["converged_scipy"]),
        "mean_posterior_certainty": np.nan if post_stats is None else post_stats["mean_posterior_certainty"],
        "mean_entropy": np.nan if post_stats is None else post_stats["mean_entropy"],
    }])

    occ_df = None
    if occ is not None:
        occ_df = pd.DataFrame({
            "state": np.arange(1, J + 1),
            "posterior_occupancy": occ,
            "hard_assignment_share": hard_share
        })

    out = {
        "fit": fit_row,
        "occupancy": occ_df,
        "emissions": em_df,
        "distances": dist_df,
        "gamma": gamma,
        "transition_matrix": A,
        "alpha": alpha,
        "beta": beta,
        "W": W,
    }
    return out

In [ ]:
# ============================================================
# RUN DIAGNOSTICS FOR CURRENT J=3 AND J=4
# ============================================================

em_names = globals().get("emission_cols", ["AI Decision Authority Share", "Escalation Share"])

diag_j3 = summarize_model(
    label="J3",
    J=3,
    model=best_p_hat_J3,
    res=best_res_J3,
    row=best_row_j3,
    emission_names=em_names
)

diag_j4 = summarize_model(
    label="J4",
    J=4,
    model=best_p_hat_J4,
    res=best_res_J4,
    row=best_row_j4,
    emission_names=em_names
)

print("=== FIT SUMMARY ===")
display(pd.concat([diag_j3["fit"], diag_j4["fit"]], axis=0).round(4))

print("\n=== J=3 OCCUPANCY ===")
display(diag_j3["occupancy"].round(4) if diag_j3["occupancy"] is not None else pd.DataFrame())

print("\n=== J=4 OCCUPANCY ===")
display(diag_j4["occupancy"].round(4) if diag_j4["occupancy"] is not None else pd.DataFrame())

print("\n=== J=3 EMISSIONS ===")
display(diag_j3["emissions"].round(4) if diag_j3["emissions"] is not None else pd.DataFrame())

print("\n=== J=4 EMISSIONS ===")
display(diag_j4["emissions"].round(4) if diag_j4["emissions"] is not None else pd.DataFrame())

print("\n=== J=3 PAIRWISE STATE DISTANCES ===")
display(diag_j3["distances"].round(4) if diag_j3["distances"] is not None else pd.DataFrame())

print("\n=== J=4 PAIRWISE STATE DISTANCES ===")
display(diag_j4["distances"].round(4) if diag_j4["distances"] is not None else pd.DataFrame())

if diag_j3["transition_matrix"] is not None:
    print("\n=== J=3 TRANSITION MATRIX ===")
    display(pd.DataFrame(diag_j3["transition_matrix"]).round(4))

if diag_j4["transition_matrix"] is not None:
    print("\n=== J=4 TRANSITION MATRIX ===")
    display(pd.DataFrame(diag_j4["transition_matrix"]).round(4))

=== FIT SUMMARY ===


,model,J,LL,AIC,BIC,soft_converged,scipy_success,mean_posterior_certainty,mean_entropy
0,J3,3,-225.8620,667.7240,1316.4117,True,True,NaN,NaN
0,J4,4,1008.8216,-1689.6432,-704.5989,True,True,NaN,NaN



=== J=3 OCCUPANCY ===


""



=== J=4 OCCUPANCY ===


...

,state,emission,mean,sigma
0,1,AI Decision Authority Share,-0.2060,NaN
1,1,Escalation Share,0.6740,NaN
2,2,AI Decision Authority Share,0.4632,NaN
3,2,Escalation Share,-0.1220,NaN
4,3,AI Decision Authority Share,-2.1908,NaN
5,3,Escalation Share,-1.9307,NaN
6,4,AI Decision Authority Share,0.2513,NaN
7,4,Escalation Share,0.1293,NaN



=== J=3 PAIRWISE STATE DISTANCES ===


,state_i,state_j,euclidean_distance_means
1,1,3,0.1875
0,1,2,3.1655
2,2,3,3.1707



=== J=4 PAIRWISE STATE DISTANCES ===


,state_i,state_j,euclidean_distance_means
4,2,4,0.3287
2,1,4,0.7112
0,1,2,1.0399
5,3,4,3.1948
3,2,3,3.2116
1,1,3,3.2747


In [20]:
# ============================================================
# OCCUPANCY TABLES FOR J=3 AND J=4
# ============================================================

import numpy as np
import pandas as pd

def _safe_getattr(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    return default

def _to_numpy(x):
    if x is None:
        return None
    try:
        return np.asarray(x)
    except Exception:
        return None

def _extract_gamma(model):
    cand = [
        "gamma", "Gamma", "posterior_probs", "posterior_probabilities",
        "smoothed_probs", "state_posteriors", "xi_gamma"
    ]
    g = _safe_getattr(model, cand, default=None)
    if g is None:
        return None

    if isinstance(g, list):
        try:
            g = np.vstack([np.asarray(a) for a in g])
        except Exception:
            return None

    g = _to_numpy(g)
    if g is None:
        return None
    return g

def make_occupancy_table(model, J, model_name="model"):
    gamma = _extract_gamma(model)
    if gamma is None:
        print(f"Could not find posterior probabilities for {model_name}.")
        return None

    if gamma.ndim != 2:
        print(f"Unexpected gamma ndim for {model_name}: {gamma.ndim}")
        return None

    if gamma.shape[1] != J and gamma.shape[0] == J:
        gamma = gamma.T

    if gamma.shape[1] != J:
        print(f"Gamma shape mismatch for {model_name}: got {gamma.shape}, expected second dim = {J}")
        return None

    occ = gamma.mean(axis=0)
    hard = gamma.argmax(axis=1)
    hard_share = np.bincount(hard, minlength=J) / len(hard)
    maxprob = gamma.max(axis=1)

    rows = []
    for j in range(J):
        state_mask = (hard == j)
        avg_cert_state = float(maxprob[state_mask].mean()) if state_mask.sum() > 0 else np.nan
        rows.append({
            "model": model_name,
            "state": j + 1,
            "posterior_occupancy": float(occ[j]),
            "hard_assignment_share": float(hard_share[j]),
            "avg_max_posterior_if_hard_assigned": avg_cert_state,
        })

    return pd.DataFrame(rows)

occ_j3 = make_occupancy_table(best_p_hat_J3, J=3, model_name="J3")
occ_j4 = make_occupancy_table(best_p_hat_J4, J=4, model_name="J4")

print("=== OCCUPANCY: J=3 ===")
display(occ_j3.round(4) if occ_j3 is not None else pd.DataFrame())

print("\n=== OCCUPANCY: J=4 ===")
display(occ_j4.round(4) if occ_j4 is not None else pd.DataFrame())

Could not find posterior probabilities for J3.
Could not find posterior probabilities for J4.
=== OCCUPANCY: J=3 ===


""



=== OCCUPANCY: J=4 ===


""


In [21]:
# ============================================================
# INSPECT FITTED OBJECT STRUCTURE
# ============================================================

def show_attrs(obj, name="obj"):
    attrs = [a for a in dir(obj) if not a.startswith("_")]
    print(f"\n=== ATTRIBUTES OF {name} ===")
    print(attrs)

show_attrs(best_p_hat_J3, "best_p_hat_J3")
show_attrs(best_p_hat_J4, "best_p_hat_J4")


=== ATTRIBUTES OF best_p_hat_J3 ===
['W', 'alpha', 'beta', 'log_sigma', 'logit_pi', 'mu']

=== ATTRIBUTES OF best_p_hat_J4 ===
['W', 'alpha', 'beta', 'log_sigma', 'logit_pi', 'mu']


In [22]:
# ============================================================
# INSPECT COMMON HMM-LIKE ATTRIBUTE NAMES
# ============================================================

import numpy as np

candidate_names = [
    "gamma", "Gamma", "posterior_probs", "posterior_probabilities",
    "smoothed_probs", "state_posteriors", "xi_gamma",
    "alpha", "beta", "log_alpha", "log_beta",
    "xi", "Xi", "A", "P", "Q",
    "mu", "Mu", "sigma", "Sigma", "W"
]

def inspect_candidates(obj, name="obj"):
    print(f"\n=== CANDIDATE ATTRIBUTES FOR {name} ===")
    for nm in candidate_names:
        if hasattr(obj, nm):
            val = getattr(obj, nm)
            try:
                arr = np.asarray(val)
                print(f"{nm:25s} shape={arr.shape} dtype={arr.dtype}")
            except Exception:
                print(f"{nm:25s} type={type(val)}")

inspect_candidates(best_p_hat_J3, "best_p_hat_J3")
inspect_candidates(best_p_hat_J4, "best_p_hat_J4")


=== CANDIDATE ATTRIBUTES FOR best_p_hat_J3 ===
alpha                     shape=(3, 3) dtype=float64
beta                      shape=(3, 3, 4) dtype=float64
mu                        shape=(3, 2) dtype=float64
W                         shape=(3, 2, 8) dtype=float64

=== CANDIDATE ATTRIBUTES FOR best_p_hat_J4 ===
alpha                     shape=(4, 4) dtype=float64
beta                      shape=(4, 4, 4) dtype=float64
mu                        shape=(4, 2) dtype=float64
W                         shape=(4, 2, 8) dtype=float64


In [23]:
# ============================================================
# INSPECT OPTIMIZER RESULT OBJECTS TOO
# ============================================================

show_attrs(best_res_J3, "best_res_J3")
show_attrs(best_res_J4, "best_res_J4")

inspect_candidates(best_res_J3, "best_res_J3")
inspect_candidates(best_res_J4, "best_res_J4")


=== ATTRIBUTES OF best_res_J3 ===
['fun', 'hess_inv', 'jac', 'k_params', 'message', 'nfev', 'nit', 'njev', 'status', 'success', 'true_ll', 'true_negll', 'x']

=== ATTRIBUTES OF best_res_J4 ===
['fun', 'hess_inv', 'jac', 'k_params', 'message', 'nfev', 'nit', 'njev', 'status', 'success', 'true_ll', 'true_negll', 'x']

=== CANDIDATE ATTRIBUTES FOR best_res_J3 ===

=== CANDIDATE ATTRIBUTES FOR best_res_J4 ===


In [24]:
# ============================================================
# FORWARD-BACKWARD POSTERIOR PROBABILITIES FOR YOUR NH-HMM
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Basic utilities
# ------------------------------------------------------------
def softmax(v):
    v = np.asarray(v, dtype=float)
    v = v - np.max(v)
    ev = np.exp(v)
    return ev / np.sum(ev)

def logsumexp(a, axis=None, keepdims=False):
    a = np.asarray(a, dtype=float)
    a_max = np.max(a, axis=axis, keepdims=True)
    out = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out

def ensure_panel(arr, name="arr"):
    """
    Convert stack into shape (N, T, K).
    Works for:
      - np.ndarray already shaped (N,T,K)
      - list of arrays [(T_i,K), ...] with equal T
    """
    if isinstance(arr, list):
        arr = [np.asarray(a) for a in arr]
        Ts = [a.shape[0] for a in arr]
        if len(set(Ts)) != 1:
            raise ValueError(f"{name} is a ragged list; this helper expects equal T.")
        return np.stack(arr, axis=0)
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"{name} must have ndim=3, got shape={arr.shape}")
    return arr

# ------------------------------------------------------------
# Extract parameters from fitted object
# ------------------------------------------------------------
def extract_params(p_hat):
    mu = np.asarray(p_hat.mu)                 # (J, D)
    W = np.asarray(p_hat.W)                   # (J, D, Kz)
    alpha = np.asarray(p_hat.alpha)           # (J, J)
    beta = np.asarray(p_hat.beta)             # (J, J, Px)
    log_sigma = np.asarray(p_hat.log_sigma)   # expect (J, D) or compatible
    logit_pi = np.asarray(p_hat.logit_pi)     # expect (J,)

    if log_sigma.ndim == 1:
        if log_sigma.shape[0] == mu.shape[0]:
            log_sigma = log_sigma.reshape(mu.shape[0], 1)
        elif log_sigma.shape[0] == mu.shape[1]:
            log_sigma = np.tile(log_sigma.reshape(1, mu.shape[1]), (mu.shape[0], 1))

    sigma = np.exp(log_sigma)

    return {
        "mu": mu,
        "W": W,
        "alpha": alpha,
        "beta": beta,
        "sigma": sigma,
        "logit_pi": logit_pi,
    }

# ------------------------------------------------------------
# Emission log-density
# ------------------------------------------------------------
def emission_loglik_per_state(y_t, z_t, params):
    """
    y_t : (D,)
    z_t : (Kz,)
    returns log p(y_t | state=j, z_t) for j=1..J   shape (J,)
    """
    mu = params["mu"]       # (J, D)
    W = params["W"]         # (J, D, Kz)
    sigma = params["sigma"] # (J, D)

    J, D = mu.shape
    out = np.zeros(J, dtype=float)

    for j in range(J):
        mean_j = mu[j] + W[j] @ z_t
        sd_j = np.clip(sigma[j], 1e-8, None)

        # diagonal Gaussian log-density
        ll_j = -0.5 * np.sum(
            np.log(2.0 * np.pi * sd_j**2) + ((y_t - mean_j) ** 2) / (sd_j**2)
        )
        out[j] = ll_j

    return out

# ------------------------------------------------------------
# Time-varying transition matrix
# ------------------------------------------------------------
def transition_matrix_from_x(x_t, params):
    """
    x_t : (Px,)
    returns A_t of shape (J, J), where rows sum to 1
    """
    alpha = params["alpha"]   # (J, J)
    beta = params["beta"]     # (J, J, Px)

    J = alpha.shape[0]
    A_t = np.zeros((J, J), dtype=float)

    for i in range(J):
        logits_i = alpha[i] + beta[i] @ x_t
        A_t[i] = softmax(logits_i)

    return A_t

# ------------------------------------------------------------
# Initial distribution
# ------------------------------------------------------------
def initial_state_probs(params):
    return softmax(params["logit_pi"])

# ------------------------------------------------------------
# Forward-backward for one sequence
# ------------------------------------------------------------
def forward_backward_one(y_seq, x_seq, z_seq, params):
    """
    y_seq : (T, D)
    x_seq : (T, Px)
    z_seq : (T, Kz)

    Transition at time t uses x_seq[t] for movement t -> t+1.
    If your implementation instead uses x_{t-1}, this is the one
    place you may need to shift indexing.
    """
    y_seq = np.asarray(y_seq)
    x_seq = np.asarray(x_seq)
    z_seq = np.asarray(z_seq)

    T = y_seq.shape[0]
    J = params["mu"].shape[0]

    logB = np.zeros((T, J), dtype=float)
    for t in range(T):
        logB[t] = emission_loglik_per_state(y_seq[t], z_seq[t], params)

    pi = initial_state_probs(params)
    log_pi = np.log(np.clip(pi, 1e-300, None))

    logA = np.zeros((T - 1, J, J), dtype=float)
    for t in range(T - 1):
        A_t = transition_matrix_from_x(x_seq[t], params)
        logA[t] = np.log(np.clip(A_t, 1e-300, None))

    # Forward
    log_alpha_fb = np.zeros((T, J), dtype=float)
    log_alpha_fb[0] = log_pi + logB[0]

    for t in range(1, T):
        for j in range(J):
            log_alpha_fb[t, j] = logB[t, j] + logsumexp(log_alpha_fb[t - 1] + logA[t - 1, :, j])

    # Backward
    log_beta_fb = np.zeros((T, J), dtype=float)
    log_beta_fb[T - 1] = 0.0

    for t in range(T - 2, -1, -1):
        for i in range(J):
            log_beta_fb[t, i] = logsumexp(logA[t, i, :] + logB[t + 1, :] + log_beta_fb[t + 1, :])

    # Smoothed posteriors gamma
    log_gamma = log_alpha_fb + log_beta_fb
    log_gamma = log_gamma - logsumexp(log_gamma, axis=1, keepdims=True)
    gamma = np.exp(log_gamma)

    # Sequence log-likelihood
    ll_seq = float(logsumexp(log_alpha_fb[-1]))

    return {
        "gamma": gamma,                  # (T, J)
        "log_alpha_fb": log_alpha_fb,    # forward probs in log-scale
        "log_beta_fb": log_beta_fb,      # backward probs in log-scale
        "logB": logB,
        "ll_seq": ll_seq,
    }

# ------------------------------------------------------------
# Apply to full panel
# ------------------------------------------------------------
def compute_posteriors_panel(p_hat, Y_stack, X_stack, Z_stack):
    params = extract_params(p_hat)

    Yp = ensure_panel(Y_stack, "Y_stack")
    Xp = ensure_panel(X_stack, "X_stack")
    Zp = ensure_panel(Z_stack, "Z_stack")

    N, T, D = Yp.shape
    J = params["mu"].shape[0]

    gamma_list = []
    ll_list = []

    for n in range(N):
        out_n = forward_backward_one(Yp[n], Xp[n], Zp[n], params)
        gamma_list.append(out_n["gamma"])
        ll_list.append(out_n["ll_seq"])

    gamma_panel = np.stack(gamma_list, axis=0)   # (N, T, J)

    return {
        "gamma_panel": gamma_panel,
        "gamma_flat": gamma_panel.reshape(-1, J),
        "ll_sum_from_fb": float(np.sum(ll_list)),
        "params": params,
    }

In [25]:
# ============================================================
# COMPUTE POSTERIORS FOR J=3 AND J=4
# ============================================================

post_j3 = compute_posteriors_panel(best_p_hat_J3, Y_stack, X_stack, Z_stack)
post_j4 = compute_posteriors_panel(best_p_hat_J4, Y_stack, X_stack, Z_stack)

print("=== FORWARD-BACKWARD LL CHECK ===")
print(f"J=3 | recomputed LL = {post_j3['ll_sum_from_fb']:.6f} | stored LL = {best_row_j3['LL']:.6f}")
print(f"J=4 | recomputed LL = {post_j4['ll_sum_from_fb']:.6f} | stored LL = {best_row_j4['LL']:.6f}")

=== FORWARD-BACKWARD LL CHECK ===
J=3 | recomputed LL = -1396.734855 | stored LL = -225.861997
J=4 | recomputed LL = -200.405692 | stored LL = 1008.821587


In [26]:
# ============================================================
# OCCUPANCY TABLES
# ============================================================

def make_occupancy_from_gamma(gamma_flat, model_name="model"):
    J = gamma_flat.shape[1]
    occ = gamma_flat.mean(axis=0)
    hard = gamma_flat.argmax(axis=1)
    hard_share = np.bincount(hard, minlength=J) / len(hard)
    maxprob = gamma_flat.max(axis=1)

    rows = []
    for j in range(J):
        mask = (hard == j)
        rows.append({
            "model": model_name,
            "state": j + 1,
            "posterior_occupancy": float(occ[j]),
            "hard_assignment_share": float(hard_share[j]),
            "avg_max_posterior_if_hard_assigned": float(maxprob[mask].mean()) if mask.sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows)

occ_j3 = make_occupancy_from_gamma(post_j3["gamma_flat"], model_name="J3")
occ_j4 = make_occupancy_from_gamma(post_j4["gamma_flat"], model_name="J4")

print("=== OCCUPANCY: J=3 ===")
display(occ_j3.round(4))

print("\n=== OCCUPANCY: J=4 ===")
display(occ_j4.round(4))

=== OCCUPANCY: J=3 ===


,model,state,posterior_occupancy,hard_assignment_share,avg_max_posterior_if_hard_assigned
0,J3,1,0.7261,0.7263,0.9765
1,J3,2,0.0920,0.0920,0.9968
2,J3,3,0.1819,0.1817,0.9080



=== OCCUPANCY: J=4 ===


,model,state,posterior_occupancy,hard_assignment_share,avg_max_posterior_if_hard_assigned
0,J4,1,0.2271,0.2293,0.8693
1,J4,2,0.2652,0.2600,0.9165
2,J4,3,0.0902,0.0900,0.9977
3,J4,4,0.4175,0.4207,0.8913


In [27]:
# ============================================================
# STATE PROFILE TABLES
# ============================================================

def make_state_profile_from_posteriors(p_hat, gamma_flat, model_name="model", emission_names=None):
    params = extract_params(p_hat)
    mu = params["mu"]
    sigma = params["sigma"]

    J, D = mu.shape
    occ = gamma_flat.mean(axis=0)
    hard = gamma_flat.argmax(axis=1)
    hard_share = np.bincount(hard, minlength=J) / len(hard)

    if emission_names is None or len(emission_names) != D:
        emission_names = [f"emission_{d+1}" for d in range(D)]

    rows = []
    for j in range(J):
        row = {
            "model": model_name,
            "state": j + 1,
            "posterior_occupancy": float(occ[j]),
            "hard_assignment_share": float(hard_share[j]),
        }
        for d in range(D):
            row[f"{emission_names[d]}_mean"] = float(mu[j, d])
            row[f"{emission_names[d]}_sigma"] = float(sigma[j, d])
        rows.append(row)

    return pd.DataFrame(rows)

emission_names = globals().get("emission_cols", ["AI Decision Authority Share", "Escalation Share"])

profile_j3 = make_state_profile_from_posteriors(
    best_p_hat_J3,
    post_j3["gamma_flat"],
    model_name="J3",
    emission_names=emission_names
)

profile_j4 = make_state_profile_from_posteriors(
    best_p_hat_J4,
    post_j4["gamma_flat"],
    model_name="J4",
    emission_names=emission_names
)

print("=== STATE PROFILE: J=3 ===")
display(profile_j3.round(4))

print("\n=== STATE PROFILE: J=4 ===")
display(profile_j4.round(4))

=== STATE PROFILE: J=3 ===


,model,state,posterior_occupancy,hard_assignment_share,AI Decision Authority Share_mean,AI Decision Authority Share_sigma,Escalation Share_mean,Escalation Share_sigma
0,J3,1,0.7261,0.7263,0.2543,0.1846,0.1272,0.2271
1,J3,2,0.0920,0.0920,-2.1557,0.5996,-1.9253,0.6339
2,J3,3,0.1819,0.1817,0.3756,0.1048,-0.0158,0.1410



=== STATE PROFILE: J=4 ===


,model,state,posterior_occupancy,hard_assignment_share,AI Decision Authority Share_mean,AI Decision Authority Share_sigma,Escalation Share_mean,Escalation Share_sigma
0,J4,1,0.2271,0.2293,-0.2060,0.0836,0.6740,0.1199
1,J4,2,0.2652,0.2600,0.4632,0.1340,-0.1220,0.1579
2,J4,3,0.0902,0.0900,-2.1908,0.5652,-1.9307,0.6403
3,J4,4,0.4175,0.4207,0.2513,0.1153,0.1293,0.1451


In [28]:
# ============================================================
# CREDIBILITY FLAGS
# ============================================================

def credibility_flags_from_profile(profile_df, min_occ=0.05, sigma_floor_warn=0.02):
    flags = []

    small_states = profile_df.loc[profile_df["posterior_occupancy"] < min_occ, "state"].tolist()
    if small_states:
        flags.append(f"Tiny occupancy state(s): {small_states}")

    sigma_cols = [c for c in profile_df.columns if c.endswith("_sigma")]
    tiny_sigma_cells = []
    for c in sigma_cols:
        bad = profile_df.loc[profile_df[c] < sigma_floor_warn, ["state", c]]
        if len(bad) > 0:
            tiny_sigma_cells.append((c, bad.values.tolist()))

    if tiny_sigma_cells:
        flags.append(f"Very small sigma detected: {tiny_sigma_cells}")

    if not flags:
        flags.append("No obvious occupancy/sigma red flags.")

    return pd.DataFrame({"flag": flags})

print("=== J=3 FLAGS ===")
display(credibility_flags_from_profile(profile_j3))

print("\n=== J=4 FLAGS ===")
display(credibility_flags_from_profile(profile_j4))

=== J=3 FLAGS ===


,flag
0,No obvious occupancy/sigma red flags.



=== J=4 FLAGS ===


,flag
0,No obvious occupancy/sigma red flags.


## Run a refit for best model

### It keeps the cross-J comparison for transparency, but sets BEST_J = 3 by design based on your theory/MISQ preference, not lowest BIC.

In [ ]:
# ============================================================
# STAGE 2: FINAL REFIT
# Main specification: J = 3
# Rationale:
#   - J=4 is statistically stronger by BIC
#   - J=3 is retained as the main model for theoretical fit,
#     interpretability, and parsimony
# Uses:
#   - out_j2["best_row"] for J=2
#   - best_row_j3 or out_j3["best_row"] for J=3
#   - best_row_j4 or out_j4["best_row"] for J=4
# Does NOT rely on older screening objects.
# ============================================================

import time
import copy
import numpy as np
import pandas as pd

# ----------------------------
# Build full cross-J comparison
# ----------------------------
_rows = []

# J=2
if "out_j2" in globals() and out_j2 is not None:
    r = out_j2["best_row"]
    _rows.append({
        "source": "final_J2",
        "J": int(r["J"]),
        "LL": float(r["LL"]),
        "AIC": float(r["AIC"]),
        "BIC": float(r["BIC"]),
        "soft_converged": bool(r["converged_soft"]),
        "scipy_success": bool(r["converged_scipy"]),
        "n_params": int(r.get("params", 0)),
    })

# J=3
if "best_row_j3" in globals() and best_row_j3 is not None:
    r = best_row_j3
    _rows.append({
        "source": "final_J3",
        "J": int(r["J"]),
        "LL": float(r["LL"]),
        "AIC": float(r["AIC"]),
        "BIC": float(r["BIC"]),
        "soft_converged": bool(r["converged_soft"]),
        "scipy_success": bool(r["converged_scipy"]),
        "n_params": int(r.get("params", 0)),
    })
elif "out_j3" in globals() and out_j3 is not None:
    r = out_j3["best_row"]
    _rows.append({
        "source": "final_J3",
        "J": int(r["J"]),
        "LL": float(r["LL"]),
        "AIC": float(r["AIC"]),
        "BIC": float(r["BIC"]),
        "soft_converged": bool(r["converged_soft"]),
        "scipy_success": bool(r["converged_scipy"]),
        "n_params": int(r.get("params", 0)),
    })

# J=4
if "best_row_j4" in globals() and best_row_j4 is not None:
    r = best_row_j4
    _rows.append({
        "source": "final_J4",
        "J": int(r["J"]),
        "LL": float(r["LL"]),
        "AIC": float(r["AIC"]),
        "BIC": float(r["BIC"]),
        "soft_converged": bool(r["converged_soft"]),
        "scipy_success": bool(r["converged_scipy"]),
        "n_params": int(r.get("params", 0)),
    })
elif "out_j4" in globals() and out_j4 is not None:
    r = out_j4["best_row"]
    _rows.append({
        "source": "final_J4",
        "J": int(r["J"]),
        "LL": float(r["LL"]),
        "AIC": float(r["AIC"]),
        "BIC": float(r["BIC"]),
        "soft_converged": bool(r["converged_soft"]),
        "scipy_success": bool(r["converged_scipy"]),
        "n_params": int(r.get("params", 0)),
    })

_df = pd.DataFrame(_rows).sort_values(["J", "BIC"]).reset_index(drop=True)

if _df.empty:
    raise RuntimeError("No final cross-J candidate rows found.")

print("Cross-J summary:")
print(_df[["J", "source", "n_params", "LL", "AIC", "BIC", "soft_converged"]].round(2).to_string(index=False))

# ----------------------------
# Theoretical main-model choice
# ----------------------------
BEST_J = 3
_rule = "theory-driven main specification (MISQ-style interpretability/parsimony)"

_j3_row = _df[_df["J"] == 3].copy()
if _j3_row.empty:
    raise RuntimeError("J=3 row not found in final cross-J comparison.")
_sel = _j3_row.iloc[0]

print(f"\nSelected main model: J={BEST_J} ({_rule})")
print(f"  source={_sel['source']}  LL={_sel['LL']:.2f}  BIC={_sel['BIC']:.2f}  converged={bool(_sel['soft_converged'])}")

# Optional note about statistical winner
_conv = _df[_df["soft_converged"] == True].copy()
if len(_conv) > 0:
    _bic_winner = _conv.loc[_conv["BIC"].idxmin()]
    print(f"\nNote: lowest-BIC converged model is J={int(_bic_winner['J'])} with BIC={float(_bic_winner['BIC']):.2f}.")
    if int(_bic_winner["J"]) != BEST_J:
        print("Main specification is retained as J=3 for theoretical interpretability; higher-J model is treated as robustness evidence.")

print(f"\n=== STAGE 2: FINAL REFIT (J={BEST_J}) ===")

# ----------------------------
# Warm start — preference order
# ----------------------------
warm_list = []

def _append_if_compatible(ws, label):
    if ws is None:
        return False
    if hasattr(ws, "W"):
        try:
            K_now = Z_stack.shape[2]
            if ws.W.shape[2] == K_now:
                warm_list.append(ws)
                print(f"✓ Warm start: {label} (K={ws.W.shape[2]})")
                return True
            else:
                print(f"⚠️ Skipping {label} (K mismatch: ws={ws.W.shape[2]}, current={K_now})")
                return False
        except Exception as e:
            print(f"⚠️ Skipping {label} (compatibility check failed: {e})")
            return False
    else:
        warm_list.append(ws)
        print(f"✓ Warm start: {label}")
        return True

if "best_p_hat_J3" in globals() and best_p_hat_J3 is not None:
    _append_if_compatible(best_p_hat_J3, "best_p_hat_J3")
elif "out_j3" in globals() and out_j3 is not None:
    _append_if_compatible(out_j3.get("best_p_hat", None), "out_j3['best_p_hat']")

if not warm_list:
    print("⚠️ No compatible warm start — random initialization")

# ----------------------------
# Final configuration for J=3
# ----------------------------
final_cfg = dict(
    maxiter=2200,
    n_starts=16,
    seed=777,
    time_cap_min=75,
    diag_bias=3.0,
    maxfun=900000,
    use_subset=False,
    l2=0.005,
    do_emission_only_warmstart=True,
    emission_only_maxiter=320,
    ftol=5e-8,
    gtol=1e-5,
)

# ----------------------------
# Run final estimation
# ----------------------------
t0 = time.time()

best_p_final, best_res_final, best_is_conv_final = fit_model_batched(
    J=BEST_J,
    Y_stack=Y_stack,
    X_stack=X_stack,
    Z_stack=Z_stack,
    sigma_min=0.08,
    sigma_max=4.0,
    print_every=50,
    warm_starts=warm_list if len(warm_list) > 0 else None,
    **final_cfg
)

elapsed = time.time() - t0

# ----------------------------
# Metrics
# ----------------------------
ll_total = float(getattr(best_res_final, "true_ll", np.nan))
k_params = len(getattr(best_res_final, "x", []))
bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
aic = 2.0 * k_params - 2.0 * ll_total

print("\n" + "="*60)
print(f"FINAL MAIN MODEL (J={BEST_J})")
print(f"LL:  {ll_total:.6f}")
print(f"AIC: {aic:.6f}")
print(f"BIC: {bic:.6f}")
print(f"Soft-converged: {bool(best_is_conv_final)}")
print(f"SciPy success:  {bool(getattr(best_res_final, 'success', False))}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print("="*60)

# ----------------------------
# Store final outputs
# ----------------------------
best_model = best_p_final
best_J = BEST_J

best_row_final = {
    "stage": f"final_refit_j{BEST_J}",
    "J": int(BEST_J),
    "params": int(k_params),
    "LL": float(ll_total),
    "AIC": float(aic),
    "BIC": float(bic),
    "converged_soft": bool(best_is_conv_final),
    "converged_scipy": bool(getattr(best_res_final, "success", False)),
    "time_s": float(elapsed),
    "message": str(getattr(best_res_final, "message", "")),
}

print("\nFinal stored objects:")
print(" - best_model")
print(" - best_J")
print(" - best_p_final")
print(" - best_res_final")
print(" - best_is_conv_final")
print(" - best_row_final")
print(" - final_cfg")

Cross-J summary:
 J   source  n_params      LL      AIC     BIC  soft_converged
 3 final_J3       108 -225.86   667.72 1316.41            True
 4 final_J4       164 1008.82 -1689.64 -704.60            True

Selected main model: J=3 (theory-driven main specification (MISQ-style interpretability/parsimony))
  source=final_J3  LL=-225.86  BIC=1316.41  converged=True

Note: lowest-BIC converged model is J=4 with BIC=-704.60.
Main specification is retained as J=3 for theoretical interpretability; higher-J model is treated as robustness evidence.

=== STAGE 2: FINAL REFIT (J=3) ===
✓ Warm start: best_p_hat_J3 (K=8)
    J=3 start 1/16 iter=50 elapsed=0.4 min
    J=3 start 1/16 iter=100 elapsed=0.7 min
    J=3 start 1/16 iter=150 elapsed=1.1 min


In [ ]:

import pickle

# Save best model artifacts for later reuse (no refit needed)

best_J = int(best_J) if "best_J" in globals() else int(globals().get("BEST_J", best_J_screen))

model_artifacts = {
    "best_model": best_model,
    "best_J": int(best_J),
    "best_res_final": globals().get("best_res_final", None),
    "final_cfg": globals().get("final_cfg", None),
    "ll_total": globals().get("ll_total", None),
    "k_params": globals().get("k_params", None),
    "n_obs_total": globals().get("n_obs_total", None),
    "aic": globals().get("aic", None),
    "bic": globals().get("bic", None),
    "emission_cols": globals().get("emission_cols", None),
    "transition_cols": globals().get("transition_cols", None),
    "control_cols": globals().get("control_cols", None),
    "label_map": globals().get("label_map", None),
    "state_order_1idx": globals().get("state_order_1idx", None),
    "y_scaler": getattr(data, "y_scaler", None),
    "x_scaler": getattr(data, "x_scaler", None),
    "z_scaler": getattr(data, "z_scaler", None),
}

out_path = Path("best_model_artifacts_dataset2_2 emissions_4covariates_03.pkl")
with out_path.open("wb") as f:
    pickle.dump(model_artifacts, f)

print(f"Saved best model artifacts → {out_path.resolve()}")


Saved best model artifacts → C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\data_analysis\best_model_artifacts_dataset2_2 emissions_4covariates_02.pkl
